<a id="chapter-15"></a>
# Chapter 15: Serialization, schemas, and explicit data boundaries

**Level:** Intermediate to advanced

**Learning goal:** Serialize domain values deliberately and reject unsupported or malformed records.

**Serialization** converts a model to a transportable representation, not to executable code.
JSON supports a limited set of primitive shapes. An ipaddress value or Enum needs an explicit
representation. Avoid dumping arbitrary __dict__ contents: internal fields, caches, and
credentials can leak, and the resulting schema changes when implementation details change.

**Deserialization is validation.** Require the fields and types your schema promises, reject
unknown schema versions, and normalize only intentionally. JSON decoding alone says nothing
about whether a VLAN, hostname, or MTU is acceptable. bool is a subclass of int in Python, so
strict numeric schemas may need a deliberate check.

**Version your records.** A schema_version field creates an explicit compatibility boundary.
Migrate old formats intentionally; do not silently treat an unsupported version as today's
format. Decide whether unknown fields are rejected, ignored, or preserved. This chapter's
small schemas prefer explicit rejection for clarity.

**Enums** give named states with a defined allowed set. Store the stable value rather than
relying on a debug repr. A redacted report should be constructed from an allowlist of public
fields. Simply deleting a key called password does not guarantee all sensitive data is gone.

**Snapshots** should express ownership. Returning a copied mapping prevents some accidental
mutation; immutable nested values or deeper copies may be needed for nested structures.
These exercises use strings, integers, tuples, and JSON records, never real secrets.

**Pause and explain:** Explain why JSON decoding is not schema validation and why dumping __dict__ is risky.

## Pattern 071: JSON round trip through explicit fields

**Core concept:** A domain model controls its public serialized schema.

**Network-engineering purpose:** An inventory record can be exported without exposing unrelated implementation attributes.

### Read the implementation in this order
1. Create a dataclass.
2. Serialize an explicit field mapping using json.dumps.
3. Decode and reconstruct through declared keys.
4. Check value equality after round trip.

**Common mistake / boundary:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

**Practice after the ten worked scenarios:** Add a from_json boundary that rejects unknown fields and wrong types.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0701 - JSON round trip through explicit fields: branch scenario

**Scenario 01/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'branch'` |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "branch-edge-01", "site": "branch"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [702]:
# Example 0701: JSON round trip through explicit fields - branch
case = {'site': 'branch', 'hostname': 'branch-edge-01'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "branch-edge-01", "site": "branch"}


### Example 0702 - JSON round trip through explicit fields: campus scenario

**Scenario 02/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'campus'` |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "campus-access-02", "site": "campus"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [703]:
# Example 0702: JSON round trip through explicit fields - campus
case = {'site': 'campus', 'hostname': 'campus-access-02'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "campus-access-02", "site": "campus"}


### Example 0703 - JSON round trip through explicit fields: dc scenario

**Scenario 03/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'dc'` |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "dc-leaf-03", "site": "dc"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [704]:
# Example 0703: JSON round trip through explicit fields - dc
case = {'site': 'dc', 'hostname': 'dc-leaf-03'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "dc-leaf-03", "site": "dc"}

### Example 0704 - JSON round trip through explicit fields: wan scenario

**Scenario 04/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wan'` |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "wan-border-04", "site": "wan"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [705]:
# Example 0704: JSON round trip through explicit fields - wan
case = {'site': 'wan', 'hostname': 'wan-border-04'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "wan-border-04", "site": "wan"}


### Example 0705 - JSON round trip through explicit fields: lab scenario

**Scenario 05/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'lab'` |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "lab-router-05", "site": "lab"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [706]:
# Example 0705: JSON round trip through explicit fields - lab
case = {'site': 'lab', 'hostname': 'lab-router-05'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "lab-router-05", "site": "lab"}


### Example 0706 - JSON round trip through explicit fields: wireless scenario

**Scenario 06/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wireless'` |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "wireless-dist-06", "site": "wireless"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [707]:
# Example 0706: JSON round trip through explicit fields - wireless
case = {'site': 'wireless', 'hostname': 'wireless-dist-06'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "wireless-dist-06", "site": "wireless"}

### Example 0707 - JSON round trip through explicit fields: remote scenario

**Scenario 07/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'remote'` |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "remote-gateway-07", "site": "remote"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [708]:
# Example 0707: JSON round trip through explicit fields - remote
case = {'site': 'remote', 'hostname': 'remote-gateway-07'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "remote-gateway-07", "site": "remote"}


### Example 0708 - JSON round trip through explicit fields: core scenario

**Scenario 08/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'core'` |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "core-router-08", "site": "core"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [709]:
# Example 0708: JSON round trip through explicit fields - core
case = {'site': 'core', 'hostname': 'core-router-08'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "core-router-08", "site": "core"}


### Example 0709 - JSON round trip through explicit fields: mgmt scenario

**Scenario 09/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'mgmt'` |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "mgmt-switch-09", "site": "mgmt"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [710]:
# Example 0709: JSON round trip through explicit fields - mgmt
case = {'site': 'mgmt', 'hostname': 'mgmt-switch-09'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "mgmt-switch-09", "site": "mgmt"}

### Example 0710 - JSON round trip through explicit fields: backup scenario

**Scenario 10/10; pattern 071; Intermediate to advanced.** This is a self-contained offline exercise. An inventory record can be exported without exposing unrelated implementation attributes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'backup'` |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Create a dataclass. Check value equality after round trip. Follow the class-specific steps under Pattern 071 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "backup-edge-10", "site": "backup"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple round trip assumes trusted shape; validate untrusted decoded data before construction.

In [711]:
# Example 0710: JSON round trip through explicit fields - backup
case = {'site': 'backup', 'hostname': 'backup-edge-10'}

import json
from dataclasses import dataclass

@dataclass
class DeviceRecord:
    hostname: str
    site: str

    def to_json(self):
        return json.dumps({"hostname": self.hostname, "site": self.site}, sort_keys=True)

original = DeviceRecord(case["hostname"], case["site"])
payload = original.to_json()
record = json.loads(payload)
restored = DeviceRecord(record["hostname"], record["site"])
assert restored == original
print(payload)


{"hostname": "backup-edge-10", "site": "backup"}


## Pattern 072: Versioned schema rejects unsupported records

**Core concept:** A version field prevents accidental interpretation of incompatible data.

**Network-engineering purpose:** A saved intent should not be applied using assumptions from a different record format.

### Read the implementation in this order
1. Require exactly the declared keys.
2. Check version and strict MTU type.
3. Validate the model's MTU range.
4. Accept a valid record and reject a newer version.

**Common mistake / boundary:** Do not silently default an unknown version to a known schema.

**Practice after the ten worked scenarios:** Implement an explicit version-0-to-version-1 migration as a separate function.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0711 - Versioned schema rejects unsupported records: branch scenario

**Scenario 01/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [712]:
# Example 0711: Versioned schema rejects unsupported records - branch
case = {'mtu': 1500}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0712 - Versioned schema rejects unsupported records: campus scenario

**Scenario 02/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [713]:
# Example 0712: Versioned schema rejects unsupported records - campus
case = {'mtu': 9000}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0713 - Versioned schema rejects unsupported records: dc scenario

**Scenario 03/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [714]:
# Example 0713: Versioned schema rejects unsupported records - dc
case = {'mtu': 9216}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0714 - Versioned schema rejects unsupported records: wan scenario

**Scenario 04/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [715]:
# Example 0714: Versioned schema rejects unsupported records - wan
case = {'mtu': 576}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0715 - Versioned schema rejects unsupported records: lab scenario

**Scenario 05/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [716]:
# Example 0715: Versioned schema rejects unsupported records - lab
case = {'mtu': 1600}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version

### Example 0716 - Versioned schema rejects unsupported records: wireless scenario

**Scenario 06/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [717]:
# Example 0716: Versioned schema rejects unsupported records - wireless
case = {'mtu': 1500}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version

### Example 0717 - Versioned schema rejects unsupported records: remote scenario

**Scenario 07/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [718]:
# Example 0717: Versioned schema rejects unsupported records - remote
case = {'mtu': 2000}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0718 - Versioned schema rejects unsupported records: core scenario

**Scenario 08/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [719]:
# Example 0718: Versioned schema rejects unsupported records - core
case = {'mtu': 4470}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0719 - Versioned schema rejects unsupported records: mgmt scenario

**Scenario 09/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [720]:
# Example 0719: Versioned schema rejects unsupported records - mgmt
case = {'mtu': 9100}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


### Example 0720 - Versioned schema rejects unsupported records: backup scenario

**Scenario 10/10; pattern 072; Intermediate to advanced.** This is a self-contained offline exercise. A saved intent should not be applied using assumptions from a different record format.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Require exactly the declared keys. Accept a valid record and reject a newer version. Follow the class-specific steps under Pattern 072 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unsupported schema version
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not silently default an unknown version to a known schema.

In [721]:
# Example 0720: Versioned schema rejects unsupported records - backup
case = {'mtu': 8192}

class IntentRecord:
    def __init__(self, mtu):
        self.mtu = mtu

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"schema_version", "mtu"}:
            raise ValueError("unexpected schema fields")
        if type(record["schema_version"]) is not int or record["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        if type(record["mtu"]) is not int or not 576 <= record["mtu"] <= 9216:
            raise ValueError("invalid MTU")
        return cls(record["mtu"])

intent = IntentRecord.from_mapping({"schema_version": 1, "mtu": case["mtu"]})
assert intent.mtu == case["mtu"]
try:
    IntentRecord.from_mapping({"schema_version": 2, "mtu": case["mtu"]})
except ValueError as error:
    assert "version" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported version accepted")


unsupported schema version


## Pattern 073: Enum represents an allowed state set

**Core concept:** Named states prevent arbitrary strings from spreading through the model.

**Network-engineering purpose:** An interface observation can represent up and down with a defined vocabulary.

### Read the implementation in this order
1. Declare an Enum with stable string values.
2. Parse an external token using the enum constructor.
3. Use the enum member in the object.
4. Reject an unrecognized token.

**Common mistake / boundary:** A real observation model may need unknown or unavailable distinct from down.

**Practice after the ten worked scenarios:** Add an UNKNOWN member and define exactly which inputs may produce it.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0721 - Enum represents an allowed state set: branch scenario

**Scenario 01/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [722]:
# Example 0721: Enum represents an allowed state set - branch
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


### Example 0722 - Enum represents an allowed state set: campus scenario

**Scenario 02/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `False` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
DOWN down
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [723]:
# Example 0722: Enum represents an allowed state set - campus
case = {'healthy': False}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected


DOWN down


### Example 0723 - Enum represents an allowed state set: dc scenario

**Scenario 03/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [724]:
# Example 0723: Enum represents an allowed state set - dc
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


### Example 0724 - Enum represents an allowed state set: wan scenario

**Scenario 04/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [725]:
# Example 0724: Enum represents an allowed state set - wan
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


### Example 0725 - Enum represents an allowed state set: lab scenario

**Scenario 05/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `False` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
DOWN down
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [726]:
# Example 0725: Enum represents an allowed state set - lab
case = {'healthy': False}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected

DOWN down


### Example 0726 - Enum represents an allowed state set: wireless scenario

**Scenario 06/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [727]:
# Example 0726: Enum represents an allowed state set - wireless
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


### Example 0727 - Enum represents an allowed state set: remote scenario

**Scenario 07/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `False` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
DOWN down
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [728]:
# Example 0727: Enum represents an allowed state set - remote
case = {'healthy': False}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
DOWN down


### Example 0728 - Enum represents an allowed state set: core scenario

**Scenario 08/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [729]:
# Example 0728: Enum represents an allowed state set - core
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


### Example 0729 - Enum represents an allowed state set: mgmt scenario

**Scenario 09/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `False` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
DOWN down
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [730]:
# Example 0729: Enum represents an allowed state set - mgmt
case = {'healthy': False}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected


DOWN down


### Example 0730 - Enum represents an allowed state set: backup scenario

**Scenario 10/10; pattern 073; Intermediate to advanced.** This is a self-contained offline exercise. An interface observation can represent up and down with a defined vocabulary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `healthy` | `True` |

**Logic to trace:** Declare an Enum with stable string values. Reject an unrecognized token. Follow the class-specific steps under Pattern 073 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown state rejected
UP up
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A real observation model may need unknown or unavailable distinct from down.

In [731]:
# Example 0730: Enum represents an allowed state set - backup
case = {'healthy': True}

from enum import Enum

class State(Enum):
    UP = "up"
    DOWN = "down"

class Observation:
    def __init__(self, state):
        self.state = State(state)

value = "up" if case["healthy"] else "down"
observation = Observation(value)
assert observation.state.value == value
try:
    Observation("mystery")
except ValueError:
    print("unknown state rejected")
else:
    raise AssertionError("unknown state accepted")
print(observation.state.name, observation.state.value)


unknown state rejected
UP up


## Pattern 074: Allowlist a public report

**Core concept:** Build the safe report from approved fields rather than copying every internal field.

**Network-engineering purpose:** An inventory object may contain internal notes that do not belong in a user-facing summary.

### Read the implementation in this order
1. Keep internal metadata separate.
2. Return only the approved hostname and site.
3. Verify the exact public key set.
4. Check internal-only data is not in the serialized report.

**Common mistake / boundary:** An allowlist limits accidental disclosure but does not replace a data-handling review.

**Practice after the ten worked scenarios:** Add a nested field and decide explicitly whether it is safe for the public report.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0731 - Allowlist a public report: branch scenario

**Scenario 01/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'branch'` |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "branch-edge-01", "site": "branch"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [732]:
# Example 0731: Allowlist a public report - branch
case = {'site': 'branch', 'hostname': 'branch-edge-01'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "branch-edge-01", "site": "branch"}


### Example 0732 - Allowlist a public report: campus scenario

**Scenario 02/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'campus'` |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "campus-access-02", "site": "campus"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [733]:
# Example 0732: Allowlist a public report - campus
case = {'site': 'campus', 'hostname': 'campus-access-02'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "campus-access-02", "site": "campus"}


### Example 0733 - Allowlist a public report: dc scenario

**Scenario 03/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'dc'` |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "dc-leaf-03", "site": "dc"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [734]:
# Example 0733: Allowlist a public report - dc
case = {'site': 'dc', 'hostname': 'dc-leaf-03'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "dc-leaf-03", "site": "dc"}

### Example 0734 - Allowlist a public report: wan scenario

**Scenario 04/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wan'` |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "wan-border-04", "site": "wan"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [735]:
# Example 0734: Allowlist a public report - wan
case = {'site': 'wan', 'hostname': 'wan-border-04'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "wan-border-04", "site": "wan"}


### Example 0735 - Allowlist a public report: lab scenario

**Scenario 05/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'lab'` |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "lab-router-05", "site": "lab"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [736]:
# Example 0735: Allowlist a public report - lab
case = {'site': 'lab', 'hostname': 'lab-router-05'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "lab-router-05", "site": "lab"}


### Example 0736 - Allowlist a public report: wireless scenario

**Scenario 06/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wireless'` |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "wireless-dist-06", "site": "wireless"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [737]:
# Example 0736: Allowlist a public report - wireless
case = {'site': 'wireless', 'hostname': 'wireless-dist-06'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "wireless-dist-06", "site": "wireless"}


### Example 0737 - Allowlist a public report: remote scenario

**Scenario 07/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'remote'` |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "remote-gateway-07", "site": "remote"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [738]:
# Example 0737: Allowlist a public report - remote
case = {'site': 'remote', 'hostname': 'remote-gateway-07'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "remote-gateway-07", "site": "remote"}


### Example 0738 - Allowlist a public report: core scenario

**Scenario 08/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'core'` |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "core-router-08", "site": "core"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [739]:
# Example 0738: Allowlist a public report - core
case = {'site': 'core', 'hostname': 'core-router-08'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "core-router-08", "site": "core"}


### Example 0739 - Allowlist a public report: mgmt scenario

**Scenario 09/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'mgmt'` |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "mgmt-switch-09", "site": "mgmt"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [740]:
# Example 0739: Allowlist a public report - mgmt
case = {'site': 'mgmt', 'hostname': 'mgmt-switch-09'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "mgmt-switch-09", "site": "mgmt"}


### Example 0740 - Allowlist a public report: backup scenario

**Scenario 10/10; pattern 074; Intermediate to advanced.** This is a self-contained offline exercise. An inventory object may contain internal notes that do not belong in a user-facing summary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'backup'` |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Keep internal metadata separate. Check internal-only data is not in the serialized report. Follow the class-specific steps under Pattern 074 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{"hostname": "backup-edge-10", "site": "backup"}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** An allowlist limits accidental disclosure but does not replace a data-handling review.

In [741]:
# Example 0740: Allowlist a public report - backup
case = {'site': 'backup', 'hostname': 'backup-edge-10'}

import json

class InventoryItem:
    def __init__(self, hostname, site):
        self.hostname, self.site = hostname, site
        self._internal_note = "lab-only-maintenance-note"

    def public_report(self):
        return {"hostname": self.hostname, "site": self.site}

item = InventoryItem(case["hostname"], case["site"])
report = item.public_report()
assert set(report) == {"hostname", "site"}
assert item._internal_note not in json.dumps(report)
print(json.dumps(report, sort_keys=True))


{"hostname": "backup-edge-10", "site": "backup"}


## Pattern 075: Snapshot ownership at a boundary

**Core concept:** A method can return independent nested data rather than a live internal mapping.

**Network-engineering purpose:** A caller editing a report should not silently change stored interface intent.

### Read the implementation in this order
1. Store the model's nested state.
2. Deep-copy only serializable configuration data for a snapshot.
3. Modify the returned snapshot.
4. Check that internal state remains at its original value.

**Common mistake / boundary:** Do not deep-copy connection objects, locks, or other external resources.

**Practice after the ten worked scenarios:** Use frozen dataclasses and tuples to design an immutable snapshot instead.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0741 - Snapshot ownership at a boundary: branch scenario

**Scenario 01/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |
| `mtu` | `1500` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet1/1': {'mtu': 1500}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [742]:
# Example 0741: Snapshot ownership at a boundary - branch
case = {'interface': 'Ethernet1/1', 'mtu': 1500}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet1/1': {'mtu': 1500}}


### Example 0742 - Snapshot ownership at a boundary: campus scenario

**Scenario 02/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |
| `mtu` | `9000` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet2/1': {'mtu': 9000}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [743]:
# Example 0742: Snapshot ownership at a boundary - campus
case = {'interface': 'Ethernet2/1', 'mtu': 9000}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet2/1': {'mtu': 9000}}


### Example 0743 - Snapshot ownership at a boundary: dc scenario

**Scenario 03/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |
| `mtu` | `9216` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet3/1': {'mtu': 9216}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [744]:
# Example 0743: Snapshot ownership at a boundary - dc
case = {'interface': 'Ethernet3/1', 'mtu': 9216}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet3/1': {'mtu': 9216}}


### Example 0744 - Snapshot ownership at a boundary: wan scenario

**Scenario 04/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |
| `mtu` | `576` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet4/1': {'mtu': 576}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [745]:
# Example 0744: Snapshot ownership at a boundary - wan
case = {'interface': 'Ethernet4/1', 'mtu': 576}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet4/1': {'mtu': 576}}


### Example 0745 - Snapshot ownership at a boundary: lab scenario

**Scenario 05/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |
| `mtu` | `1600` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet5/1': {'mtu': 1600}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [746]:
# Example 0745: Snapshot ownership at a boundary - lab
case = {'interface': 'Ethernet5/1', 'mtu': 1600}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet5/1': {'mtu': 1600}}


### Example 0746 - Snapshot ownership at a boundary: wireless scenario

**Scenario 06/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |
| `mtu` | `1500` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet6/1': {'mtu': 1500}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [747]:
# Example 0746: Snapshot ownership at a boundary - wireless
case = {'interface': 'Ethernet6/1', 'mtu': 1500}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet6/1': {'mtu': 1500}}


### Example 0747 - Snapshot ownership at a boundary: remote scenario

**Scenario 07/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |
| `mtu` | `2000` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet7/1': {'mtu': 2000}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [748]:
# Example 0747: Snapshot ownership at a boundary - remote
case = {'interface': 'Ethernet7/1', 'mtu': 2000}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet7/1': {'mtu': 2000}}


### Example 0748 - Snapshot ownership at a boundary: core scenario

**Scenario 08/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |
| `mtu` | `4470` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet8/1': {'mtu': 4470}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [749]:
# Example 0748: Snapshot ownership at a boundary - core
case = {'interface': 'Ethernet8/1', 'mtu': 4470}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet8/1': {'mtu': 4470}}


### Example 0749 - Snapshot ownership at a boundary: mgmt scenario

**Scenario 09/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |
| `mtu` | `9100` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet9/1': {'mtu': 9100}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [750]:
# Example 0749: Snapshot ownership at a boundary - mgmt
case = {'interface': 'Ethernet9/1', 'mtu': 9100}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet9/1': {'mtu': 9100}}


### Example 0750 - Snapshot ownership at a boundary: backup scenario

**Scenario 10/10; pattern 075; Intermediate to advanced.** This is a self-contained offline exercise. A caller editing a report should not silently change stored interface intent.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |
| `mtu` | `8192` |

**Logic to trace:** Store the model's nested state. Check that internal state remains at its original value. Follow the class-specific steps under Pattern 075 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
{'Ethernet10/1': {'mtu': 8192}}
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not deep-copy connection objects, locks, or other external resources.

In [751]:
# Example 0750: Snapshot ownership at a boundary - backup
case = {'interface': 'Ethernet10/1', 'mtu': 8192}

from copy import deepcopy

class Configuration:
    def __init__(self, interface, mtu):
        self._state = {interface: {"mtu": mtu}}

    def snapshot(self):
        return deepcopy(self._state)

configuration = Configuration(case["interface"], case["mtu"])
snapshot = configuration.snapshot()
snapshot[case["interface"]]["mtu"] = 1
assert configuration.snapshot()[case["interface"]]["mtu"] == case["mtu"]
print(configuration.snapshot())


{'Ethernet10/1': {'mtu': 8192}}


### Chapter 15 review

**Explain without running:** Explain why JSON decoding is not schema validation and why dumping __dict__ is risky.

**Hands-on review:** pick one pattern from this chapter, change its scenario to a boundary condition, predict the output, and add one assertion that would catch an incorrect implementation. Name the invariant, the object that owns its state, and the exception behavior. Then explain which parts are offline teaching assumptions and which parts would need a reviewed real-device adapter.

<a id="chapter-16"></a>
# Chapter 16: Advanced object construction and attribute mechanics

**Level:** Advanced

**Learning goal:** Understand descriptors, slots, caching, __new__, and class-creation hooks without overusing them.

**Descriptors** define attribute behavior through methods such as __get__, __set__, and
__set_name__. Properties and ordinary bound methods rely on descriptor mechanisms. A
reusable validating descriptor can enforce the same invariant across several fields. Access
through the class should normally return the descriptor itself when instance is None.

**Attribute lookup** is richer than 'instance dictionary then class'. Data descriptors can
take precedence over instance dictionary entries; non-data descriptors such as ordinary
methods participate differently. __getattr__ is a fallback for missing attributes.
__getattribute__ intercepts ordinary attribute access and is easy to make recursively broken.
Prefer a property or explicit method unless interception is truly required.

**__slots__** declares a set of attribute storage names for instances and can remove their
ordinary __dict__ when the hierarchy permits it. It is not a security boundary, deep
immutability, or a universal performance fix. A base class with a dictionary can change the
result. Weak references and inheritance require deliberate design.

**Cached properties** compute once and store a result. They suit expensive derived values only
when invalidation semantics are clear. A cached value can become stale if inputs change.
Concurrent access may also compute a value more than once, so do not rely on exactly-once
side effects.

**Creation hooks.** __new__ creates instances and is useful when subclassing immutable built-ins.
__init_subclass__ customizes subclass creation without requiring a custom metaclass.
A metaclass controls class creation itself; `type(SomeClass)` is its metaclass. Most network
automation should prefer ordinary classes and explicit registries over metaclass machinery.

**Pause and explain:** Choose a property before a descriptor unless reuse justifies it; explain cache invalidation.

## Pattern 076: Reusable validating descriptor

**Core concept:** One descriptor can validate assignment to a declared model field.

**Network-engineering purpose:** Several numeric configuration fields can share range-checking machinery.

### Read the implementation in this order
1. Capture a backing name with __set_name__.
2. Return self for access through the class.
3. Validate before writing through instance storage.
4. Check valid construction and preserved state after rejection.

**Common mistake / boundary:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

**Practice after the ten worked scenarios:** Add a second descriptor field with a different range and test both independently.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0751 - Reusable validating descriptor: branch scenario

**Scenario 01/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [752]:
# Example 0751: Reusable validating descriptor - branch
case = {'mtu': 1500}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


1500


### Example 0752 - Reusable validating descriptor: campus scenario

**Scenario 02/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
9000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [753]:
# Example 0752: Reusable validating descriptor - campus
case = {'mtu': 9000}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


9000


### Example 0753 - Reusable validating descriptor: dc scenario

**Scenario 03/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
9216
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [754]:
# Example 0753: Reusable validating descriptor - dc
case = {'mtu': 9216}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


9216


### Example 0754 - Reusable validating descriptor: wan scenario

**Scenario 04/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
576
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [755]:
# Example 0754: Reusable validating descriptor - wan
case = {'mtu': 576}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


576


### Example 0755 - Reusable validating descriptor: lab scenario

**Scenario 05/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
1600
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [756]:
# Example 0755: Reusable validating descriptor - lab
case = {'mtu': 1600}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


1600


### Example 0756 - Reusable validating descriptor: wireless scenario

**Scenario 06/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
1500
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [757]:
# Example 0756: Reusable validating descriptor - wireless
case = {'mtu': 1500}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


1500


### Example 0757 - Reusable validating descriptor: remote scenario

**Scenario 07/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
2000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [758]:
# Example 0757: Reusable validating descriptor - remote
case = {'mtu': 2000}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


2000

### Example 0758 - Reusable validating descriptor: core scenario

**Scenario 08/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
4470
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [759]:
# Example 0758: Reusable validating descriptor - core
case = {'mtu': 4470}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


4470


### Example 0759 - Reusable validating descriptor: mgmt scenario

**Scenario 09/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
9100
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [760]:
# Example 0759: Reusable validating descriptor - mgmt
case = {'mtu': 9100}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


9100

### Example 0760 - Reusable validating descriptor: backup scenario

**Scenario 10/10; pattern 076; Advanced.** This is a self-contained offline exercise. Several numeric configuration fields can share range-checking machinery.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Capture a backing name with __set_name__. Check valid construction and preserved state after rejection. Follow the class-specific steps under Pattern 076 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
8192
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This descriptor uses __dict__; it would need a different storage design for dictionary-free slots.

In [761]:
# Example 0760: Reusable validating descriptor - backup
case = {'mtu': 8192}

class BoundedInteger:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.storage = "_" + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        if type(value) is not int or not self.low <= value <= self.high:
            raise ValueError("outside declared range")
        instance.__dict__[self.storage] = value

class Interface:
    mtu = BoundedInteger(576, 9216)

    def __init__(self, mtu):
        self.mtu = mtu

interface = Interface(case["mtu"])
try:
    interface.mtu = 0
except ValueError:
    assert interface.mtu == case["mtu"]
else:
    raise AssertionError("descriptor accepted invalid value")
assert isinstance(Interface.mtu, BoundedInteger)
print(interface.mtu)


8192

## Pattern 077: Slots catch accidental field names

**Core concept:** A dictionary-free slotted object restricts ordinary instance attribute names.

**Network-engineering purpose:** A misspelled hostname field can fail immediately instead of silently creating a new attribute.

### Read the implementation in this order
1. Declare slots on a direct object subclass.
2. Initialize the declared field.
3. Try writing a misspelled field.
4. Verify AttributeError and the original field.

**Common mistake / boundary:** Slots do not make values immutable and do not replace validation or access control.

**Practice after the ten worked scenarios:** Compare vars() behavior for a normal instance and a slotted instance.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0761 - Slots catch accidental field names: branch scenario

**Scenario 01/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [762]:
# Example 0761: Slots catch accidental field names - branch
case = {'hostname': 'branch-edge-01'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected

### Example 0762 - Slots catch accidental field names: campus scenario

**Scenario 02/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [763]:
# Example 0762: Slots catch accidental field names - campus
case = {'hostname': 'campus-access-02'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0763 - Slots catch accidental field names: dc scenario

**Scenario 03/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [764]:
# Example 0763: Slots catch accidental field names - dc
case = {'hostname': 'dc-leaf-03'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0764 - Slots catch accidental field names: wan scenario

**Scenario 04/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [765]:
# Example 0764: Slots catch accidental field names - wan
case = {'hostname': 'wan-border-04'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0765 - Slots catch accidental field names: lab scenario

**Scenario 05/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [766]:
# Example 0765: Slots catch accidental field names - lab
case = {'hostname': 'lab-router-05'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0766 - Slots catch accidental field names: wireless scenario

**Scenario 06/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [767]:
# Example 0766: Slots catch accidental field names - wireless
case = {'hostname': 'wireless-dist-06'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0767 - Slots catch accidental field names: remote scenario

**Scenario 07/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [768]:
# Example 0767: Slots catch accidental field names - remote
case = {'hostname': 'remote-gateway-07'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0768 - Slots catch accidental field names: core scenario

**Scenario 08/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [769]:
# Example 0768: Slots catch accidental field names - core
case = {'hostname': 'core-router-08'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0769 - Slots catch accidental field names: mgmt scenario

**Scenario 09/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [770]:
# Example 0769: Slots catch accidental field names - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


### Example 0770 - Slots catch accidental field names: backup scenario

**Scenario 10/10; pattern 077; Advanced.** This is a self-contained offline exercise. A misspelled hostname field can fail immediately instead of silently creating a new attribute.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Declare slots on a direct object subclass. Verify AttributeError and the original field. Follow the class-specific steps under Pattern 077 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
misspelled field rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Slots do not make values immutable and do not replace validation or access control.

In [771]:
# Example 0770: Slots catch accidental field names - backup
case = {'hostname': 'backup-edge-10'}

class Device:
    __slots__ = ("hostname",)

    def __init__(self, hostname):
        self.hostname = hostname

device = Device(case["hostname"])
try:
    device.hostnmae = "typo"
except AttributeError:
    print("misspelled field rejected")
else:
    raise AssertionError("unexpected attribute accepted")
assert device.hostname == case["hostname"]
assert not hasattr(device, "__dict__")


misspelled field rejected


## Pattern 078: Cached derived value and invalidation

**Core concept:** cached_property stores a computed result until its cached attribute is deleted.

**Network-engineering purpose:** A normalized inventory summary can be cached, but changes to source data must invalidate it.

### Read the implementation in this order
1. Define the cached property.
2. Read it once to populate the cache.
3. Change the input and observe the old cached value.
4. Delete the cache entry and verify recomputation.

**Common mistake / boundary:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

**Practice after the ten worked scenarios:** Replace manual deletion with an update method that invalidates automatically.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0771 - Cached derived value and invalidation: branch scenario

**Scenario 01/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [772]:
# Example 0771: Cached derived value and invalidation - branch
case = {'hostname': 'branch-edge-01'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed

### Example 0772 - Cached derived value and invalidation: campus scenario

**Scenario 02/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [773]:
# Example 0772: Cached derived value and invalidation - campus
case = {'hostname': 'campus-access-02'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed

### Example 0773 - Cached derived value and invalidation: dc scenario

**Scenario 03/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [774]:
# Example 0773: Cached derived value and invalidation - dc
case = {'hostname': 'dc-leaf-03'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed

### Example 0774 - Cached derived value and invalidation: wan scenario

**Scenario 04/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [775]:
# Example 0774: Cached derived value and invalidation - wan
case = {'hostname': 'wan-border-04'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed

### Example 0775 - Cached derived value and invalidation: lab scenario

**Scenario 05/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [776]:
# Example 0775: Cached derived value and invalidation - lab
case = {'hostname': 'lab-router-05'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed


### Example 0776 - Cached derived value and invalidation: wireless scenario

**Scenario 06/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [777]:
# Example 0776: Cached derived value and invalidation - wireless
case = {'hostname': 'wireless-dist-06'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed


### Example 0777 - Cached derived value and invalidation: remote scenario

**Scenario 07/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [778]:
# Example 0777: Cached derived value and invalidation - remote
case = {'hostname': 'remote-gateway-07'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed


### Example 0778 - Cached derived value and invalidation: core scenario

**Scenario 08/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [779]:
# Example 0778: Cached derived value and invalidation - core
case = {'hostname': 'core-router-08'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed


### Example 0779 - Cached derived value and invalidation: mgmt scenario

**Scenario 09/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [780]:
# Example 0779: Cached derived value and invalidation - mgmt
case = {'hostname': 'mgmt-switch-09'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed


### Example 0780 - Cached derived value and invalidation: backup scenario

**Scenario 10/10; pattern 078; Advanced.** This is a self-contained offline exercise. A normalized inventory summary can be cached, but changes to source data must invalidate it.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Define the cached property. Delete the cache entry and verify recomputation. Follow the class-specific steps under Pattern 078 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
changed
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Caching mutable or externally changing observations without expiry can return dangerously stale data.

In [781]:
# Example 0780: Cached derived value and invalidation - backup
case = {'hostname': 'backup-edge-10'}

from functools import cached_property

class InventoryLabel:
    def __init__(self, hostname):
        self.hostname = hostname

    @cached_property
    def normalized(self):
        return self.hostname.lower()

label = InventoryLabel(case["hostname"])
assert label.normalized == case["hostname"]
label.hostname = "CHANGED"
assert label.normalized == case["hostname"]
del label.normalized
assert label.normalized == "changed"
print(label.normalized)


changed


## Pattern 079: Validate an immutable value in __new__

**Core concept:** An immutable built-in subclass must choose its value during instance creation.

**Network-engineering purpose:** A small lab VLAN value can behave numerically while rejecting invalid construction.

### Read the implementation in this order
1. Override __new__ on an int subclass.
2. Validate before creating the immutable integer.
3. Delegate construction to super().__new__.
4. Check numeric behavior and invalid-value rejection.

**Common mistake / boundary:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

**Practice after the ten worked scenarios:** Use composition instead and compare equality semantics with ordinary integers.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0781 - Validate an immutable value in __new__: branch scenario

**Scenario 01/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `1` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [782]:
# Example 0781: Validate an immutable value in __new__ - branch
case = {'vlan': 1}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
1


### Example 0782 - Validate an immutable value in __new__: campus scenario

**Scenario 02/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `10` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [783]:
# Example 0782: Validate an immutable value in __new__ - campus
case = {'vlan': 10}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
10


### Example 0783 - Validate an immutable value in __new__: dc scenario

**Scenario 03/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `100` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
100
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [784]:
# Example 0783: Validate an immutable value in __new__ - dc
case = {'vlan': 100}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction


100


### Example 0784 - Validate an immutable value in __new__: wan scenario

**Scenario 04/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `200` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
200
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [785]:
# Example 0784: Validate an immutable value in __new__ - wan
case = {'vlan': 200}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
200


### Example 0785 - Validate an immutable value in __new__: lab scenario

**Scenario 05/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `999` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
999
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [786]:
# Example 0785: Validate an immutable value in __new__ - lab
case = {'vlan': 999}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction


999


### Example 0786 - Validate an immutable value in __new__: wireless scenario

**Scenario 06/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `1000` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
1000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [787]:
# Example 0786: Validate an immutable value in __new__ - wireless
case = {'vlan': 1000}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
1000


### Example 0787 - Validate an immutable value in __new__: remote scenario

**Scenario 07/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `2000` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
2000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [788]:
# Example 0787: Validate an immutable value in __new__ - remote
case = {'vlan': 2000}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction


2000


### Example 0788 - Validate an immutable value in __new__: core scenario

**Scenario 08/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `3000` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
3000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [789]:
# Example 0788: Validate an immutable value in __new__ - core
case = {'vlan': 3000}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction


3000


### Example 0789 - Validate an immutable value in __new__: mgmt scenario

**Scenario 09/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `4000` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
4000
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [790]:
# Example 0789: Validate an immutable value in __new__ - mgmt
case = {'vlan': 4000}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction
4000


### Example 0790 - Validate an immutable value in __new__: backup scenario

**Scenario 10/10; pattern 079; Advanced.** This is a self-contained offline exercise. A small lab VLAN value can behave numerically while rejecting invalid construction.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `4093` |

**Logic to trace:** Override __new__ on an int subclass. Check numeric behavior and invalid-value rejection. Follow the class-specific steps under Pattern 079 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
immutable value rejected invalid construction
4093
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Subclassing built-ins is advanced; a frozen wrapper dataclass is often clearer for a domain value.

In [791]:
# Example 0790: Validate an immutable value in __new__ - backup
case = {'vlan': 4093}

class VlanId(int):
    def __new__(cls, value):
        if type(value) is not int or not 1 <= value <= 4094:
            raise ValueError("invalid lab VLAN")
        return super().__new__(cls, value)

vlan = VlanId(case["vlan"])
assert int(vlan) == case["vlan"]
assert vlan + 1 == case["vlan"] + 1
try:
    VlanId(0)
except ValueError:
    print("immutable value rejected invalid construction")
else:
    raise AssertionError("invalid VLAN created")
print(vlan)


immutable value rejected invalid construction


4093


## Pattern 080: Register subclasses with __init_subclass__

**Core concept:** A subclass creation hook can enforce an extension invariant.

**Network-engineering purpose:** A controlled plugin family can require unique short names at definition time.

### Read the implementation in this order
1. Define the registry on a local base class.
2. Accept a named class-creation keyword.
3. Call super's hook and reject duplicate names.
4. Check the concrete class was registered and inspect its metaclass.

**Common mistake / boundary:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

**Practice after the ten worked scenarios:** Write the same registry using an explicit register call and compare discoverability.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0791 - Register subclasses with __init_subclass__: branch scenario

**Scenario 01/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [792]:
# Example 0791: Register subclasses with __init_subclass__ - branch
case = {'mtu': 1500}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0792 - Register subclasses with __init_subclass__: campus scenario

**Scenario 02/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [793]:
# Example 0792: Register subclasses with __init_subclass__ - campus
case = {'mtu': 9000}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0793 - Register subclasses with __init_subclass__: dc scenario

**Scenario 03/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [794]:
# Example 0793: Register subclasses with __init_subclass__ - dc
case = {'mtu': 9216}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0794 - Register subclasses with __init_subclass__: wan scenario

**Scenario 04/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [795]:
# Example 0794: Register subclasses with __init_subclass__ - wan
case = {'mtu': 576}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0795 - Register subclasses with __init_subclass__: lab scenario

**Scenario 05/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [796]:
# Example 0795: Register subclasses with __init_subclass__ - lab
case = {'mtu': 1600}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0796 - Register subclasses with __init_subclass__: wireless scenario

**Scenario 06/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [797]:
# Example 0796: Register subclasses with __init_subclass__ - wireless
case = {'mtu': 1500}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu']

 type


### Example 0797 - Register subclasses with __init_subclass__: remote scenario

**Scenario 07/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [798]:
# Example 0797: Register subclasses with __init_subclass__ - remote
case = {'mtu': 2000}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu']

 type


### Example 0798 - Register subclasses with __init_subclass__: core scenario

**Scenario 08/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [799]:
# Example 0798: Register subclasses with __init_subclass__ - core
case = {'mtu': 4470}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0799 - Register subclasses with __init_subclass__: mgmt scenario

**Scenario 09/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [800]:
# Example 0799: Register subclasses with __init_subclass__ - mgmt
case = {'mtu': 9100}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Example 0800 - Register subclasses with __init_subclass__: backup scenario

**Scenario 10/10; pattern 080; Advanced.** This is a self-contained offline exercise. A controlled plugin family can require unique short names at definition time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Define the registry on a local base class. Check the concrete class was registered and inspect its metaclass. Follow the class-specific steps under Pattern 080 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mtu'] type
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Implicit import-time registration can hide dependencies; explicit registration is usually easier to audit.

In [801]:
# Example 0800: Register subclasses with __init_subclass__ - backup
case = {'mtu': 8192}

class Check:
    registry = {}

    def __init_subclass__(cls, *, name, **kwargs):
        super().__init_subclass__(**kwargs)
        if name in Check.registry:
            raise ValueError("duplicate plugin name")
        Check.registry[name] = cls

class MtuCheck(Check, name="mtu"):
    def run(self, value):
        return 576 <= value <= 9216

assert Check.registry["mtu"] is MtuCheck
assert Check.registry["mtu"]().run(case["mtu"])
assert type(MtuCheck) is type
print(sorted(Check.registry), type(MtuCheck).__name__)


['mtu'] type


### Chapter 16 review

**Explain without running:** Choose a property before a descriptor unless reuse justifies it; explain cache invalidation.

**Hands-on review:** pick one pattern from this chapter, change its scenario to a boundary condition, predict the output, and add one assertion that would catch an incorrect implementation. Name the invariant, the object that owns its state, and the exception behavior. Then explain which parts are offline teaching assumptions and which parts would need a reviewed real-device adapter.

<a id="chapter-17"></a>
# Chapter 17: Concurrency and asynchronous object APIs

**Level:** Advanced

**Learning goal:** Bound concurrent work, preserve errors, and avoid blocking the notebook event loop.

**Concurrency** means tasks can make progress over overlapping periods. It is not the same as
CPU parallelism. asyncio is cooperative: a coroutine gives other tasks opportunities to run
at awaits. A normal blocking transport call inside async def still blocks the event loop.
Use an asynchronous adapter or move appropriate blocking I/O to a bounded thread pool.

**Notebook event loops.** Jupyter usually already has an event loop. These cells use top-level
await rather than asyncio.run inside the notebook. In a normal script, put equivalent work
in async main() and call asyncio.run(main()) once at the entry point. Each async cell defines
all its classes and imports itself.

**Structured concurrency.** A TaskGroup owns a collection of tasks; exiting its context waits
for them and handles task failures according to its contract. Do not create detached tasks
and forget their results. Cancellation is a control signal; cleanup belongs in finally or an
async context manager, and CancelledError should normally propagate after cleanup.

**Bound work.** A semaphore limits simultaneous operations, not total queue size. A large
inventory may also need a bounded producer/consumer queue. Timeouts and rate limits address
different problems. Avoid launching a task for every device in a very large fleet at once.

**Shared state.** Threads need synchronization around compound updates. A Lock can protect an
invariant, but keep critical sections short and do not hold a regular blocking lock across
an await. ThreadPoolExecutor is suitable for bounded blocking I/O; ordinary CPU-bound Python
work and interpreter build details require separate performance reasoning. These lessons
use tiny local workloads and no real network, real sleeps, or long-running background tasks.

**Pause and explain:** Explain top-level await, why blocking I/O still blocks inside async def, and what a semaphore limits.

## Pattern 081: Async method gathers offline observations

**Core concept:** An async object API can coordinate awaitable operations.

**Network-engineering purpose:** A poller can collect several simulated device outcomes while preserving result order.

### Read the implementation in this order
1. Define async read on the poller.
2. Yield with asyncio.sleep(0) without a real delay.
3. Gather two coroutine results.
4. Assert output order follows input order.

**Common mistake / boundary:** async def alone does not make blocking libraries nonblocking.

**Practice after the ten worked scenarios:** Replace gather with explicit tasks and inspect each task's result after completion.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0801 - Async method gathers offline observations: branch scenario

**Scenario 01/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['branch-edge-01:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [802]:
# Example 0801: Async method gathers offline observations - branch
case = {'hostname': 'branch-edge-01'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['branch-edge-01:up', 'lab-spare:up']


### Example 0802 - Async method gathers offline observations: campus scenario

**Scenario 02/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['campus-access-02:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [803]:
# Example 0802: Async method gathers offline observations - campus
case = {'hostname': 'campus-access-02'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['campus-access-02:up', 'lab-spare:up']


### Example 0803 - Async method gathers offline observations: dc scenario

**Scenario 03/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['dc-leaf-03:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [804]:
# Example 0803: Async method gathers offline observations - dc
case = {'hostname': 'dc-leaf-03'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['dc-leaf-03:up', 'lab-spare:up']

### Example 0804 - Async method gathers offline observations: wan scenario

**Scenario 04/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wan-border-04:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [805]:
# Example 0804: Async method gathers offline observations - wan
case = {'hostname': 'wan-border-04'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['wan-border-04:up', 'lab-spare:up']

### Example 0805 - Async method gathers offline observations: lab scenario

**Scenario 05/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['lab-router-05:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [806]:
# Example 0805: Async method gathers offline observations - lab
case = {'hostname': 'lab-router-05'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['lab-router-05:up', 'lab-spare:up']

### Example 0806 - Async method gathers offline observations: wireless scenario

**Scenario 06/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wireless-dist-06:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [807]:
# Example 0806: Async method gathers offline observations - wireless
case = {'hostname': 'wireless-dist-06'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['wireless-dist-06:up', 'lab-spare:up']

### Example 0807 - Async method gathers offline observations: remote scenario

**Scenario 07/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['remote-gateway-07:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [808]:
# Example 0807: Async method gathers offline observations - remote
case = {'hostname': 'remote-gateway-07'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['remote-gateway-07:up', 'lab-spare:up']

### Example 0808 - Async method gathers offline observations: core scenario

**Scenario 08/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['core-router-08:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [809]:
# Example 0808: Async method gathers offline observations - core
case = {'hostname': 'core-router-08'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['core-router-08:up', 'lab-spare:up']

### Example 0809 - Async method gathers offline observations: mgmt scenario

**Scenario 09/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mgmt-switch-09:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [810]:
# Example 0809: Async method gathers offline observations - mgmt
case = {'hostname': 'mgmt-switch-09'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['mgmt-switch-09:up', 'lab-spare:up']


### Example 0810 - Async method gathers offline observations: backup scenario

**Scenario 10/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['backup-edge-10:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [811]:
# Example 0810: Async method gathers offline observations - backup
case = {'hostname': 'backup-edge-10'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['backup-edge-10:up', 'lab-spare:up']


## Pattern 082: Semaphore bounds in-flight operations

**Core concept:** A semaphore guards a concurrent section around each operation.

**Network-engineering purpose:** A simulated poller should never exceed its configured in-flight budget.

### Read the implementation in this order
1. Create one shared semaphore.
2. Track active and peak counts inside its async context.
3. Use finally to decrement active.
4. Gather a small workload and assert the measured peak is within the limit.

**Common mistake / boundary:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

**Practice after the ten worked scenarios:** Use a bounded queue and a fixed number of workers for a much larger input.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0811 - Semaphore bounds in-flight operations: branch scenario

**Scenario 01/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |
| `concurrency` | `1` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 1 limit: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [812]:
# Example 0811: Semaphore bounds in-flight operations - branch
case = {'hostname': 'branch-edge-01', 'concurrency': 1}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 1 limit: 1


### Example 0812 - Semaphore bounds in-flight operations: campus scenario

**Scenario 02/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |
| `concurrency` | `2` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 2 limit: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [813]:
# Example 0812: Semaphore bounds in-flight operations - campus
case = {'hostname': 'campus-access-02', 'concurrency': 2}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 2 limit: 2


### Example 0813 - Semaphore bounds in-flight operations: dc scenario

**Scenario 03/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |
| `concurrency` | `3` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 3 limit: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [814]:
# Example 0813: Semaphore bounds in-flight operations - dc
case = {'hostname': 'dc-leaf-03', 'concurrency': 3}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 3 limit: 3


### Example 0814 - Semaphore bounds in-flight operations: wan scenario

**Scenario 04/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |
| `concurrency` | `1` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 1 limit: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [815]:
# Example 0814: Semaphore bounds in-flight operations - wan
case = {'hostname': 'wan-border-04', 'concurrency': 1}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak:

 1 limit: 1


### Example 0815 - Semaphore bounds in-flight operations: lab scenario

**Scenario 05/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |
| `concurrency` | `2` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 2 limit: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [816]:
# Example 0815: Semaphore bounds in-flight operations - lab
case = {'hostname': 'lab-router-05', 'concurrency': 2}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak:

 2 limit: 2


### Example 0816 - Semaphore bounds in-flight operations: wireless scenario

**Scenario 06/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |
| `concurrency` | `3` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 3 limit: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [817]:
# Example 0816: Semaphore bounds in-flight operations - wireless
case = {'hostname': 'wireless-dist-06', 'concurrency': 3}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak:

 3 limit: 3


### Example 0817 - Semaphore bounds in-flight operations: remote scenario

**Scenario 07/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |
| `concurrency` | `1` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 1 limit: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [818]:
# Example 0817: Semaphore bounds in-flight operations - remote
case = {'hostname': 'remote-gateway-07', 'concurrency': 1}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak:

 1 limit: 1


### Example 0818 - Semaphore bounds in-flight operations: core scenario

**Scenario 08/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |
| `concurrency` | `2` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 2 limit: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [819]:
# Example 0818: Semaphore bounds in-flight operations - core
case = {'hostname': 'core-router-08', 'concurrency': 2}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak:

 2 limit: 2


### Example 0819 - Semaphore bounds in-flight operations: mgmt scenario

**Scenario 09/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |
| `concurrency` | `3` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 3 limit: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [820]:
# Example 0819: Semaphore bounds in-flight operations - mgmt
case = {'hostname': 'mgmt-switch-09', 'concurrency': 3}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak:

 3 limit: 3


### Example 0820 - Semaphore bounds in-flight operations: backup scenario

**Scenario 10/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |
| `concurrency` | `1` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 1 limit: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [821]:
# Example 0820: Semaphore bounds in-flight operations - backup
case = {'hostname': 'backup-edge-10', 'concurrency': 1}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 1 limit: 1


## Pattern 083: TaskGroup owns concurrent tasks

**Core concept:** A structured task scope keeps task lifetime tied to the calling operation.

**Network-engineering purpose:** A small fleet collection waits for every child before building its final report.

### Read the implementation in this order
1. Define an async collector method.
2. Create tasks inside TaskGroup.
3. Read results only after the group exits.
4. Check the report contains all expected values.

**Common mistake / boundary:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

**Practice after the ten worked scenarios:** Make one task raise a domain error and inspect the resulting ExceptionGroup in a separate exercise.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0821 - TaskGroup owns concurrent tasks: branch scenario

**Scenario 01/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'branch-edge-01', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [822]:
# Example 0821: TaskGroup owns concurrent tasks - branch
case = {'hostname': 'branch-edge-01'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'branch-edge-01', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0822 - TaskGroup owns concurrent tasks: campus scenario

**Scenario 02/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'campus-access-02', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [823]:
# Example 0822: TaskGroup owns concurrent tasks - campus
case = {'hostname': 'campus-access-02'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'campus-access-02', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0823 - TaskGroup owns concurrent tasks: dc scenario

**Scenario 03/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'dc-leaf-03', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [824]:
# Example 0823: TaskGroup owns concurrent tasks - dc
case = {'hostname': 'dc-leaf-03'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'dc-leaf-03', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0824 - TaskGroup owns concurrent tasks: wan scenario

**Scenario 04/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'wan-border-04', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [825]:
# Example 0824: TaskGroup owns concurrent tasks - wan
case = {'hostname': 'wan-border-04'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'wan-border-04', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0825 - TaskGroup owns concurrent tasks: lab scenario

**Scenario 05/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'lab-router-05', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [826]:
# Example 0825: TaskGroup owns concurrent tasks - lab
case = {'hostname': 'lab-router-05'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'lab-router-05', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0826 - TaskGroup owns concurrent tasks: wireless scenario

**Scenario 06/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'wireless-dist-06', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [827]:
# Example 0826: TaskGroup owns concurrent tasks - wireless
case = {'hostname': 'wireless-dist-06'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'wireless-dist-06', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0827 - TaskGroup owns concurrent tasks: remote scenario

**Scenario 07/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'remote-gateway-07', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [828]:
# Example 0827: TaskGroup owns concurrent tasks - remote
case = {'hostname': 'remote-gateway-07'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'remote-gateway-07', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]

### Example 0828 - TaskGroup owns concurrent tasks: core scenario

**Scenario 08/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'core-router-08', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [829]:
# Example 0828: TaskGroup owns concurrent tasks - core
case = {'hostname': 'core-router-08'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'core-router-08', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0829 - TaskGroup owns concurrent tasks: mgmt scenario

**Scenario 09/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'mgmt-switch-09', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [830]:
# Example 0829: TaskGroup owns concurrent tasks - mgmt
case = {'hostname': 'mgmt-switch-09'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'mgmt-switch-09', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0830 - TaskGroup owns concurrent tasks: backup scenario

**Scenario 10/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'backup-edge-10', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [831]:
# Example 0830: TaskGroup owns concurrent tasks - backup
case = {'hostname': 'backup-edge-10'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'backup-edge-10', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]

## Pattern 084: Lock protects a shared threaded counter

**Core concept:** A lock protects the complete read-modify-write operation.

**Network-engineering purpose:** A metrics collector should not rely on incidental interpreter behavior for compound updates.

### Read the implementation in this order
1. Store a Lock on the counter.
2. Increment while holding it.
3. Run a bounded batch through a context-managed executor.
4. Assert the final total equals the submitted work.

**Common mistake / boundary:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

**Practice after the ten worked scenarios:** Track successes and failures under one lock while preserving a total-attempt invariant.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0831 - Lock protects a shared threaded counter: branch scenario

**Scenario 01/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `6` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [832]:
# Example 0831: Lock protects a shared threaded counter - branch
case = {'jobs': 6}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 6


### Example 0832 - Lock protects a shared threaded counter: campus scenario

**Scenario 02/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `7` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 7
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [833]:
# Example 0832: Lock protects a shared threaded counter - campus
case = {'jobs': 7}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 7


### Example 0833 - Lock protects a shared threaded counter: dc scenario

**Scenario 03/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `8` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 8
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [834]:
# Example 0833: Lock protects a shared threaded counter - dc
case = {'jobs': 8}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 8


### Example 0834 - Lock protects a shared threaded counter: wan scenario

**Scenario 04/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `9` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 9
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [835]:
# Example 0834: Lock protects a shared threaded counter - wan
case = {'jobs': 9}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed:

 9


### Example 0835 - Lock protects a shared threaded counter: lab scenario

**Scenario 05/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `10` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [836]:
# Example 0835: Lock protects a shared threaded counter - lab
case = {'jobs': 10}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 10


### Example 0836 - Lock protects a shared threaded counter: wireless scenario

**Scenario 06/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `11` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 11
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [837]:
# Example 0836: Lock protects a shared threaded counter - wireless
case = {'jobs': 11}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 11


### Example 0837 - Lock protects a shared threaded counter: remote scenario

**Scenario 07/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `12` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 12
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [838]:
# Example 0837: Lock protects a shared threaded counter - remote
case = {'jobs': 12}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 12


### Example 0838 - Lock protects a shared threaded counter: core scenario

**Scenario 08/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `13` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 13
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [839]:
# Example 0838: Lock protects a shared threaded counter - core
case = {'jobs': 13}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 13


### Example 0839 - Lock protects a shared threaded counter: mgmt scenario

**Scenario 09/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `14` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 14
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [840]:
# Example 0839: Lock protects a shared threaded counter - mgmt
case = {'jobs': 14}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 14


### Example 0840 - Lock protects a shared threaded counter: backup scenario

**Scenario 10/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `15` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 15
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [841]:
# Example 0840: Lock protects a shared threaded counter - backup
case = {'jobs': 15}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 15


## Pattern 085: Bounded executor preserves per-item failures

**Core concept:** Future results must be inspected so worker errors reach the caller.

**Network-engineering purpose:** A fleet report can distinguish one failed simulated read from successful devices.

### Read the implementation in this order
1. Submit known targets to a small pool.
2. Associate each future with its target.
3. Read every result and catch only the expected TimeoutError.
4. Check success and failure counts and the failed hostname.

**Common mistake / boundary:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

**Practice after the ten worked scenarios:** Return a structured result dataclass instead of separate success and failure lists.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0841 - Bounded executor preserves per-item failures: branch scenario

**Scenario 01/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['branch-edge-01:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [842]:
# Example 0841: Bounded executor preserves per-item failures - branch
case = {'hostname': 'branch-edge-01'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['branch-edge-01:up'] failure: ['lab-failed']


### Example 0842 - Bounded executor preserves per-item failures: campus scenario

**Scenario 02/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['campus-access-02:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [843]:
# Example 0842: Bounded executor preserves per-item failures - campus
case = {'hostname': 'campus-access-02'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['campus-access-02:up'] failure: ['lab-failed']


### Example 0843 - Bounded executor preserves per-item failures: dc scenario

**Scenario 03/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['dc-leaf-03:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [844]:
# Example 0843: Bounded executor preserves per-item failures - dc
case = {'hostname': 'dc-leaf-03'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['dc-leaf-03:up'] failure: ['lab-failed']


### Example 0844 - Bounded executor preserves per-item failures: wan scenario

**Scenario 04/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['wan-border-04:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [845]:
# Example 0844: Bounded executor preserves per-item failures - wan
case = {'hostname': 'wan-border-04'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success:

 ['wan-border-04:up'] failure: ['lab-failed']


### Example 0845 - Bounded executor preserves per-item failures: lab scenario

**Scenario 05/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['lab-router-05:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [846]:
# Example 0845: Bounded executor preserves per-item failures - lab
case = {'hostname': 'lab-router-05'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['lab-router-05:up'] failure: ['lab-failed']


### Example 0846 - Bounded executor preserves per-item failures: wireless scenario

**Scenario 06/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['wireless-dist-06:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [847]:
# Example 0846: Bounded executor preserves per-item failures - wireless
case = {'hostname': 'wireless-dist-06'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['wireless-dist-06:up'] failure: ['lab-failed']


### Example 0847 - Bounded executor preserves per-item failures: remote scenario

**Scenario 07/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['remote-gateway-07:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [848]:
# Example 0847: Bounded executor preserves per-item failures - remote
case = {'hostname': 'remote-gateway-07'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['remote-gateway-07:up'] failure: ['lab-failed']


### Example 0848 - Bounded executor preserves per-item failures: core scenario

**Scenario 08/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['core-router-08:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [849]:
# Example 0848: Bounded executor preserves per-item failures - core
case = {'hostname': 'core-router-08'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['core-router-08:up'] failure: ['lab-failed']


### Example 0849 - Bounded executor preserves per-item failures: mgmt scenario

**Scenario 09/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['mgmt-switch-09:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [850]:
# Example 0849: Bounded executor preserves per-item failures - mgmt
case = {'hostname': 'mgmt-switch-09'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['mgmt-switch-09:up'] failure: ['lab-failed']


### Example 0850 - Bounded executor preserves per-item failures: backup scenario

**Scenario 10/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['backup-edge-10:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [851]:
# Example 0850: Bounded executor preserves per-item failures - backup
case = {'hostname': 'backup-edge-10'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['backup-edge-10:up'] failure: ['lab-failed']


### Chapter 17 review

**Explain without running:** Explain top-level await, why blocking I/O still blocks inside async def, and what a semaphore limits.

**Hands-on review:** pick one pattern from this chapter, change its scenario to a boundary condition, predict the output, and add one assertion that would catch an incorrect implementation. Name the invariant, the object that owns its state, and the exception behavior. Then explain which parts are offline teaching assumptions and which parts would need a reviewed real-device adapter.

<a id="chapter-18"></a>
# Chapter 18: Telemetry, counters, and bounded state

**Level:** Advanced

**Learning goal:** Model units, counter deltas, explicit rollover assumptions, windows, and priority.

**Always name units.** Bytes, bits, seconds, milliseconds, percentages, and fractions are not
interchangeable. For a monotonically increasing byte counter, a basic rate is
`(new_bytes - old_bytes) * 8 / elapsed_seconds`. Reject nonpositive elapsed time. A negative
delta is not a negative traffic rate; it may mean reset, rollover, an out-of-order sample,
or a different interface identity.

**Rollover requires assumptions.** A modulo calculation works only when the counter width is
known, there has been no reset, and the observation interval permits at most the number of
wraps your model can distinguish. Two samples alone generally cannot prove those assumptions.
The rollover exercise states a one-wrap, no-reset lab assumption explicitly.

**Sliding windows** retain recent samples using bounded storage such as deque(maxlen=...).
An average smooths variation but can hide spikes and depends on sample intervals. These
examples use equally weighted samples; irregular telemetry may need time weighting.

**Health decisions** need clear boundaries: does a load equal to the limit pass? Is missing
data unknown or unhealthy? A label such as OK should be backed by explicit conditions.
Hysteresis can avoid flapping when values oscillate near a threshold.

**Priority queues** can schedule more important checks first. Include a stable sequence
tie-breaker if task objects themselves are not orderable. Priority ordering is not fairness,
rate limiting, or bounded total work; a production scheduler must address those separately.

**Pause and explain:** Compute units for a rate and explain why two samples cannot distinguish every reset from rollover.

## Pattern 086: Byte counters to bits per second

**Core concept:** A value object can make a unit conversion explicit and validated.

**Network-engineering purpose:** An interface rate should not silently treat bytes as bits or accept zero elapsed time.

### Read the implementation in this order
1. Validate positive elapsed seconds.
2. Reject a negative delta as reset or unordered input.
3. Convert byte delta to bits and divide by time.
4. Check a known arithmetic result.

**Common mistake / boundary:** This simple rate assumes both counters describe the same interface and no wrap occurred.

**Practice after the ten worked scenarios:** Return a structured unavailable result for resets rather than inventing a traffic rate.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0851 - Byte counters to bits per second: branch scenario

**Scenario 01/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `1000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [852]:
# Example 0851: Byte counters to bits per second - branch
case = {'counter': 1000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0852 - Byte counters to bits per second: campus scenario

**Scenario 02/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `2000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [853]:
# Example 0852: Byte counters to bits per second - campus
case = {'counter': 2000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0853 - Byte counters to bits per second: dc scenario

**Scenario 03/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `3000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [854]:
# Example 0853: Byte counters to bits per second - dc
case = {'counter': 3000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0854 - Byte counters to bits per second: wan scenario

**Scenario 04/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `4000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [855]:
# Example 0854: Byte counters to bits per second - wan
case = {'counter': 4000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0855 - Byte counters to bits per second: lab scenario

**Scenario 05/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `5000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [856]:
# Example 0855: Byte counters to bits per second - lab
case = {'counter': 5000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0856 - Byte counters to bits per second: wireless scenario

**Scenario 06/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `6000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [857]:
# Example 0856: Byte counters to bits per second - wireless
case = {'counter': 6000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0857 - Byte counters to bits per second: remote scenario

**Scenario 07/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `7000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [858]:
# Example 0857: Byte counters to bits per second - remote
case = {'counter': 7000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0858 - Byte counters to bits per second: core scenario

**Scenario 08/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `8000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [859]:
# Example 0858: Byte counters to bits per second - core
case = {'counter': 8000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0859 - Byte counters to bits per second: mgmt scenario

**Scenario 09/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `9000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [860]:
# Example 0859: Byte counters to bits per second - mgmt
case = {'counter': 9000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


### Example 0860 - Byte counters to bits per second: backup scenario

**Scenario 10/10; pattern 086; Advanced.** This is a self-contained offline exercise. An interface rate should not silently treat bytes as bits or accept zero elapsed time.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `10000` |

**Logic to trace:** Validate positive elapsed seconds. Check a known arithmetic result. Follow the class-specific steps under Pattern 086 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
zero interval rejected
bps: 4000.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple rate assumes both counters describe the same interface and no wrap occurred.

In [861]:
# Example 0860: Byte counters to bits per second - backup
case = {'counter': 10000}

class RateCalculator:
    def bits_per_second(self, previous, current, seconds):
        if seconds <= 0:
            raise ValueError("elapsed seconds must be positive")
        if current < previous:
            raise ValueError("counter reset or unordered sample")
        return (current - previous) * 8 / seconds

calculator = RateCalculator()
rate = calculator.bits_per_second(case["counter"], case["counter"] + 1000, 2)
assert rate == 4000
try:
    calculator.bits_per_second(10, 20, 0)
except ValueError:
    print("zero interval rejected")
else:
    raise AssertionError("zero interval accepted")
print("bps:", rate)


zero interval rejected
bps: 4000.0


## Pattern 087: Counter rollover under an explicit assumption

**Core concept:** Modulo arithmetic can model one known-width wrap in a controlled lab.

**Network-engineering purpose:** A small synthetic counter makes rollover reasoning easy to inspect.

### Read the implementation in this order
1. Store the modulus from the bit width.
2. Validate both samples fit the width.
3. Compute the modular difference.
4. Test a deliberate wrap near the maximum.

**Common mistake / boundary:** A reset or multiple wraps cannot be identified from this result alone.

**Practice after the ten worked scenarios:** Add a maximum plausible delta and report ambiguity instead of claiming a reliable rate.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0861 - Counter rollover under an explicit assumption: branch scenario

**Scenario 01/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `0` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [862]:
# Example 0861: Counter rollover under an explicit assumption - branch
case = {'small_counter': 0}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 6


### Example 0862 - Counter rollover under an explicit assumption: campus scenario

**Scenario 02/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `1` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 7
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [863]:
# Example 0862: Counter rollover under an explicit assumption - campus
case = {'small_counter': 1}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 7


### Example 0863 - Counter rollover under an explicit assumption: dc scenario

**Scenario 03/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `2` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 8
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [864]:
# Example 0863: Counter rollover under an explicit assumption - dc
case = {'small_counter': 2}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 8


### Example 0864 - Counter rollover under an explicit assumption: wan scenario

**Scenario 04/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `3` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 9
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [865]:
# Example 0864: Counter rollover under an explicit assumption - wan
case = {'small_counter': 3}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 9


### Example 0865 - Counter rollover under an explicit assumption: lab scenario

**Scenario 05/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `4` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [866]:
# Example 0865: Counter rollover under an explicit assumption - lab
case = {'small_counter': 4}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 10


### Example 0866 - Counter rollover under an explicit assumption: wireless scenario

**Scenario 06/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `5` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 11
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [867]:
# Example 0866: Counter rollover under an explicit assumption - wireless
case = {'small_counter': 5}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 11


### Example 0867 - Counter rollover under an explicit assumption: remote scenario

**Scenario 07/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `6` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 12
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [868]:
# Example 0867: Counter rollover under an explicit assumption - remote
case = {'small_counter': 6}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 12


### Example 0868 - Counter rollover under an explicit assumption: core scenario

**Scenario 08/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `7` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 13
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [869]:
# Example 0868: Counter rollover under an explicit assumption - core
case = {'small_counter': 7}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta:

 13


### Example 0869 - Counter rollover under an explicit assumption: mgmt scenario

**Scenario 09/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `8` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 14
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [870]:
# Example 0869: Counter rollover under an explicit assumption - mgmt
case = {'small_counter': 8}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 14


### Example 0870 - Counter rollover under an explicit assumption: backup scenario

**Scenario 10/10; pattern 087; Advanced.** This is a self-contained offline exercise. A small synthetic counter makes rollover reasoning easy to inspect.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `small_counter` | `9` |

**Logic to trace:** Store the modulus from the bit width. Test a deliberate wrap near the maximum. Follow the class-specific steps under Pattern 087 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
one-wrap/no-reset lab delta: 15
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A reset or multiple wraps cannot be identified from this result alone.

In [871]:
# Example 0870: Counter rollover under an explicit assumption - backup
case = {'small_counter': 9}

class WrappingCounter:
    def __init__(self, bits):
        if type(bits) is not int or bits < 1:
            raise ValueError("invalid width")
        self.modulus = 1 << bits

    def delta_assuming_no_reset(self, previous, current):
        if not 0 <= previous < self.modulus or not 0 <= current < self.modulus:
            raise ValueError("sample outside width")
        return (current - previous) % self.modulus

counter = WrappingCounter(8)
previous = 250
current = case["small_counter"]
delta = counter.delta_assuming_no_reset(previous, current)
assert delta == 6 + current
print("one-wrap/no-reset lab delta:", delta)


one-wrap/no-reset lab delta: 15


## Pattern 088: Bounded sliding sample window

**Core concept:** A deque limits retained history while the object exposes a useful query.

**Network-engineering purpose:** A telemetry collector can keep a fixed number of recent observations without unbounded growth.

### Read the implementation in this order
1. Initialize deque with a positive maxlen.
2. Append more samples than the window size.
3. Compute a mean only when data exists.
4. Check only the most recent samples remain.

**Common mistake / boundary:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

**Practice after the ten worked scenarios:** Expose minimum and maximum without returning the internal deque.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0871 - Bounded sliding sample window: branch scenario

**Scenario 01/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `20` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [872]:
# Example 0871: Bounded sliding sample window - branch
case = {'load': 20}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0872 - Bounded sliding sample window: campus scenario

**Scenario 02/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `90` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [873]:
# Example 0872: Bounded sliding sample window - campus
case = {'load': 90}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0873 - Bounded sliding sample window: dc scenario

**Scenario 03/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `80` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [874]:
# Example 0873: Bounded sliding sample window - dc
case = {'load': 80}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0874 - Bounded sliding sample window: wan scenario

**Scenario 04/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `0` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [875]:
# Example 0874: Bounded sliding sample window - wan
case = {'load': 0}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0875 - Bounded sliding sample window: lab scenario

**Scenario 05/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `100` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [876]:
# Example 0875: Bounded sliding sample window - lab
case = {'load': 100}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0876 - Bounded sliding sample window: wireless scenario

**Scenario 06/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `65` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [877]:
# Example 0876: Bounded sliding sample window - wireless
case = {'load': 65}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0877 - Bounded sliding sample window: remote scenario

**Scenario 07/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `71` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [878]:
# Example 0877: Bounded sliding sample window - remote
case = {'load': 71}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0878 - Bounded sliding sample window: core scenario

**Scenario 08/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `50` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [879]:
# Example 0878: Bounded sliding sample window - core
case = {'load': 50}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0879 - Bounded sliding sample window: mgmt scenario

**Scenario 09/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `95` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [880]:
# Example 0879: Bounded sliding sample window - mgmt
case = {'load': 95}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


### Example 0880 - Bounded sliding sample window: backup scenario

**Scenario 10/10; pattern 088; Advanced.** This is a self-contained offline exercise. A telemetry collector can keep a fixed number of recent observations without unbounded growth.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `load` | `10` |

**Logic to trace:** Initialize deque with a positive maxlen. Check only the most recent samples remain. Follow the class-specific steps under Pattern 088 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
recent mean: 20.0
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a sample-weighted mean, not a time-weighted mean for irregular polling intervals.

In [881]:
# Example 0880: Bounded sliding sample window - backup
case = {'load': 10}

from collections import deque

class Window:
    def __init__(self, size):
        if size < 1:
            raise ValueError("window must be positive")
        self.samples = deque(maxlen=size)

    def add(self, value):
        self.samples.append(value)

    def mean(self):
        if not self.samples:
            raise LookupError("no samples")
        return sum(self.samples) / len(self.samples)

window = Window(3)
for value in [case["load"], 10, 20, 30]:
    window.add(value)
assert list(window.samples) == [10, 20, 30]
assert window.mean() == 20
print("recent mean:", window.mean())


recent mean: 20.0


## Pattern 089: Hysteresis avoids threshold flapping

**Core concept:** Different enter and clear thresholds make a stateful alarm deliberate.

**Network-engineering purpose:** A utilization alarm can remain active until load falls far enough below the alarm boundary.

### Read the implementation in this order
1. Validate low is below high.
2. Enter alarm only at the high threshold.
3. Retain the state between thresholds.
4. Clear at the low threshold and verify the transition sequence.

**Common mistake / boundary:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

**Practice after the ten worked scenarios:** Add an UNKNOWN state for stale observations rather than forcing a healthy result.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0881 - Hysteresis avoids threshold flapping: branch scenario

**Scenario 01/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `80` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [882]:
# Example 0881: Hysteresis avoids threshold flapping - branch
case = {'threshold': 80}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0882 - Hysteresis avoids threshold flapping: campus scenario

**Scenario 02/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `80` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [883]:
# Example 0882: Hysteresis avoids threshold flapping - campus
case = {'threshold': 80}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]

### Example 0883 - Hysteresis avoids threshold flapping: dc scenario

**Scenario 03/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `80` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [884]:
# Example 0883: Hysteresis avoids threshold flapping - dc
case = {'threshold': 80}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0884 - Hysteresis avoids threshold flapping: wan scenario

**Scenario 04/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `60` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [885]:
# Example 0884: Hysteresis avoids threshold flapping - wan
case = {'threshold': 60}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0885 - Hysteresis avoids threshold flapping: lab scenario

**Scenario 05/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `90` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [886]:
# Example 0885: Hysteresis avoids threshold flapping - lab
case = {'threshold': 90}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]

### Example 0886 - Hysteresis avoids threshold flapping: wireless scenario

**Scenario 06/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `65` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [887]:
# Example 0886: Hysteresis avoids threshold flapping - wireless
case = {'threshold': 65}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0887 - Hysteresis avoids threshold flapping: remote scenario

**Scenario 07/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `70` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [888]:
# Example 0887: Hysteresis avoids threshold flapping - remote
case = {'threshold': 70}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0888 - Hysteresis avoids threshold flapping: core scenario

**Scenario 08/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `75` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [889]:
# Example 0888: Hysteresis avoids threshold flapping - core
case = {'threshold': 75}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0889 - Hysteresis avoids threshold flapping: mgmt scenario

**Scenario 09/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `85` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [890]:
# Example 0889: Hysteresis avoids threshold flapping - mgmt
case = {'threshold': 85}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


### Example 0890 - Hysteresis avoids threshold flapping: backup scenario

**Scenario 10/10; pattern 089; Advanced.** This is a self-contained offline exercise. A utilization alarm can remain active until load falls far enough below the alarm boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `threshold` | `50` |

**Logic to trace:** Validate low is below high. Clear at the low threshold and verify the transition sequence. Follow the class-specific steps under Pattern 089 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[True, True, False]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Hysteresis does not replace missing-data handling, debounce timing, or alert ownership.

In [891]:
# Example 0890: Hysteresis avoids threshold flapping - backup
case = {'threshold': 50}

class Alarm:
    def __init__(self, low, high):
        if low >= high:
            raise ValueError("invalid hysteresis")
        self.low, self.high, self.active = low, high, False

    def update(self, load):
        if load >= self.high:
            self.active = True
        elif load <= self.low:
            self.active = False
        return self.active

high = case["threshold"]
alarm = Alarm(high - 10, high)
states = [alarm.update(value) for value in [high, high - 5, high - 10]]
assert states == [True, True, False]
print(states)


[True, True, False]


## Pattern 090: Priority queue with a stable tie-breaker

**Core concept:** An explicit sequence number avoids comparing non-orderable task objects.

**Network-engineering purpose:** Equal-priority health checks should keep a predictable submission order.

### Read the implementation in this order
1. Store priority, sequence, and payload in each heap item.
2. Increment sequence per submission.
3. Pop through the queue API.
4. Check priority order and stability for a tie.

**Common mistake / boundary:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

**Practice after the ten worked scenarios:** Add an aging policy that eventually promotes waiting checks.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0891 - Priority queue with a stable tie-breaker: branch scenario

**Scenario 01/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['branch-edge-01', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [892]:
# Example 0891: Priority queue with a stable tie-breaker - branch
case = {'hostname': 'branch-edge-01'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['branch-edge-01', 'second-critical', 'routine']


### Example 0892 - Priority queue with a stable tie-breaker: campus scenario

**Scenario 02/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['campus-access-02', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [893]:
# Example 0892: Priority queue with a stable tie-breaker - campus
case = {'hostname': 'campus-access-02'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['campus-access-02', 'second-critical', 'routine']


### Example 0893 - Priority queue with a stable tie-breaker: dc scenario

**Scenario 03/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['dc-leaf-03', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [894]:
# Example 0893: Priority queue with a stable tie-breaker - dc
case = {'hostname': 'dc-leaf-03'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['dc-leaf-03', 'second-critical', 'routine']


### Example 0894 - Priority queue with a stable tie-breaker: wan scenario

**Scenario 04/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wan-border-04', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [895]:
# Example 0894: Priority queue with a stable tie-breaker - wan
case = {'hostname': 'wan-border-04'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['wan-border-04', 'second-critical', 'routine']


### Example 0895 - Priority queue with a stable tie-breaker: lab scenario

**Scenario 05/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['lab-router-05', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [896]:
# Example 0895: Priority queue with a stable tie-breaker - lab
case = {'hostname': 'lab-router-05'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['lab-router-05', 'second-critical', 'routine']


### Example 0896 - Priority queue with a stable tie-breaker: wireless scenario

**Scenario 06/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wireless-dist-06', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [897]:
# Example 0896: Priority queue with a stable tie-breaker - wireless
case = {'hostname': 'wireless-dist-06'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['wireless-dist-06', 'second-critical', 'routine']


### Example 0897 - Priority queue with a stable tie-breaker: remote scenario

**Scenario 07/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['remote-gateway-07', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [898]:
# Example 0897: Priority queue with a stable tie-breaker - remote
case = {'hostname': 'remote-gateway-07'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['remote-gateway-07', 'second-critical', 'routine']

### Example 0898 - Priority queue with a stable tie-breaker: core scenario

**Scenario 08/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['core-router-08', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [899]:
# Example 0898: Priority queue with a stable tie-breaker - core
case = {'hostname': 'core-router-08'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['core-router-08', 'second-critical', 'routine']


### Example 0899 - Priority queue with a stable tie-breaker: mgmt scenario

**Scenario 09/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mgmt-switch-09', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [900]:
# Example 0899: Priority queue with a stable tie-breaker - mgmt
case = {'hostname': 'mgmt-switch-09'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['mgmt-switch-09', 'second-critical', 'routine']


### Example 0900 - Priority queue with a stable tie-breaker: backup scenario

**Scenario 10/10; pattern 090; Advanced.** This is a self-contained offline exercise. Equal-priority health checks should keep a predictable submission order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Store priority, sequence, and payload in each heap item. Check priority order and stability for a tie. Follow the class-specific steps under Pattern 090 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['backup-edge-10', 'second-critical', 'routine']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A priority queue can starve low-priority work; fairness is a separate scheduling requirement.

In [901]:
# Example 0900: Priority queue with a stable tie-breaker - backup
case = {'hostname': 'backup-edge-10'}

import heapq
from itertools import count

class Scheduler:
    def __init__(self):
        self.queue, self.sequence = [], count()

    def add(self, priority, payload):
        heapq.heappush(self.queue, (priority, next(self.sequence), payload))

    def pop(self):
        return heapq.heappop(self.queue)[2]

scheduler = Scheduler()
scheduler.add(5, "routine")
scheduler.add(1, case["hostname"])
scheduler.add(1, "second-critical")
result = [scheduler.pop() for _ in range(3)]
assert result == [case["hostname"], "second-critical", "routine"]
print(result)


['backup-edge-10', 'second-critical', 'routine']


### Chapter 18 review

**Explain without running:** Compute units for a rate and explain why two samples cannot distinguish every reset from rollover.

**Hands-on review:** pick one pattern from this chapter, change its scenario to a boundary condition, predict the output, and add one assertion that would catch an incorrect implementation. Name the invariant, the object that owns its state, and the exception behavior. Then explain which parts are offline teaching assumptions and which parts would need a reviewed real-device adapter.

<a id="chapter-19"></a>
# Chapter 19: Topology and route-analysis objects

**Level:** Advanced

**Learning goal:** Encapsulate small graph algorithms and state the limits of a topology model.

**A topology graph** represents nodes and edges, not the whole operational network.
Decide whether links are directed, weighted, parallel, or associated with interfaces. The
simple undirected graph here stores neighbor sets and assumes one logical link between any
pair of different nodes. A missing path in this graph means the model has no path, not proof
that a real device is isolated.

**Breadth-first search (BFS)** finds a path with the fewest edges in an unweighted graph.
Use deque, record visited nodes, and avoid revisiting them. Sorted neighbors make tiny lab
examples deterministic. Weighted cost requires a different algorithm such as Dijkstra when
weights are nonnegative.

**Cycle detection** must respect graph type. For an undirected graph, the edge back to the
immediate parent is not itself a cycle. A directed graph needs a different treatment of the
active recursion path. A graph cycle does not automatically imply an Ethernet forwarding loop:
protocols and forwarding state determine actual behavior.

**Redundancy analysis** can remove one modeled link and check reachability again. It is only
a simplified single-failure exercise. Real resilience includes shared-risk groups, power,
control-plane convergence, capacity, fate-sharing, and failure detection.

**Address planning** benefits from value objects and overlap checks. Overlapping prefixes are
not always an error: nested routes are normal, while unintended overlap in the same address
allocation scope can be a problem. Scope and policy determine interpretation.

**Pause and explain:** Explain the difference between modeled connectivity and actual forwarding reachability.

## Pattern 091: Encapsulated undirected adjacency

**Core concept:** A graph object maintains reciprocal adjacency through one operation.

**Network-engineering purpose:** Topology callers should not have to remember to add both directions themselves.

### Read the implementation in this order
1. Initialize an adjacency mapping.
2. Add both directions when linking two different nodes.
3. Expose an immutable sorted neighbor view.
4. Check reciprocity and deduplication.

**Common mistake / boundary:** This simple graph intentionally omits parallel links and interface-level metadata.

**Practice after the ten worked scenarios:** Represent each link as an immutable value with two endpoint interfaces.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0901 - Encapsulated undirected adjacency: branch scenario

**Scenario 01/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('branch-edge-01',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [902]:
# Example 0901: Encapsulated undirected adjacency - branch
case = {'hostname': 'branch-edge-01'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('branch-edge-01',)


### Example 0902 - Encapsulated undirected adjacency: campus scenario

**Scenario 02/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('campus-access-02',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [903]:
# Example 0902: Encapsulated undirected adjacency - campus
case = {'hostname': 'campus-access-02'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('campus-access-02',)


### Example 0903 - Encapsulated undirected adjacency: dc scenario

**Scenario 03/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('dc-leaf-03',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [904]:
# Example 0903: Encapsulated undirected adjacency - dc
case = {'hostname': 'dc-leaf-03'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('dc-leaf-03',)


### Example 0904 - Encapsulated undirected adjacency: wan scenario

**Scenario 04/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('wan-border-04',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [905]:
# Example 0904: Encapsulated undirected adjacency - wan
case = {'hostname': 'wan-border-04'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('wan-border-04',)


### Example 0905 - Encapsulated undirected adjacency: lab scenario

**Scenario 05/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('lab-router-05',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [906]:
# Example 0905: Encapsulated undirected adjacency - lab
case = {'hostname': 'lab-router-05'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('lab-router-05',)


### Example 0906 - Encapsulated undirected adjacency: wireless scenario

**Scenario 06/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('wireless-dist-06',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [907]:
# Example 0906: Encapsulated undirected adjacency - wireless
case = {'hostname': 'wireless-dist-06'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('wireless-dist-06',)


### Example 0907 - Encapsulated undirected adjacency: remote scenario

**Scenario 07/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('remote-gateway-07',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [908]:
# Example 0907: Encapsulated undirected adjacency - remote
case = {'hostname': 'remote-gateway-07'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('remote-gateway-07',)


### Example 0908 - Encapsulated undirected adjacency: core scenario

**Scenario 08/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('core-router-08',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [909]:
# Example 0908: Encapsulated undirected adjacency - core
case = {'hostname': 'core-router-08'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('core-router-08',)


### Example 0909 - Encapsulated undirected adjacency: mgmt scenario

**Scenario 09/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('mgmt-switch-09',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [910]:
# Example 0909: Encapsulated undirected adjacency - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('mgmt-switch-09',)


### Example 0910 - Encapsulated undirected adjacency: backup scenario

**Scenario 10/10; pattern 091; Advanced.** This is a self-contained offline exercise. Topology callers should not have to remember to add both directions themselves.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Initialize an adjacency mapping. Check reciprocity and deduplication. Follow the class-specific steps under Pattern 091 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
('backup-edge-10',)
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This simple graph intentionally omits parallel links and interface-level metadata.

In [911]:
# Example 0910: Encapsulated undirected adjacency - backup
case = {'hostname': 'backup-edge-10'}

class Topology:
    def __init__(self):
        self.adjacency = {}

    def link(self, first, second):
        if first == second:
            raise ValueError("self links not supported")
        self.adjacency.setdefault(first, set()).add(second)
        self.adjacency.setdefault(second, set()).add(first)

    def neighbors(self, node):
        return tuple(sorted(self.adjacency[node]))

topology = Topology()
topology.link(case["hostname"], "core")
topology.link(case["hostname"], "core")
assert topology.neighbors(case["hostname"]) == ("core",)
assert topology.neighbors("core") == (case["hostname"],)
print(topology.neighbors("core"))


('backup-edge-10',)


## Pattern 092: Breadth-first shortest hop path

**Core concept:** BFS explores increasing hop distance and remembers visited nodes.

**Network-engineering purpose:** An offline topology model can explain a shortest unweighted path.

### Read the implementation in this order
1. Put the source path in a deque.
2. Pop paths from the front.
3. Visit sorted unvisited neighbors.
4. Return the first destination path or raise when unreachable.

**Common mistake / boundary:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

**Practice after the ten worked scenarios:** Track predecessors instead of copying full paths for better scaling.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0911 - Breadth-first shortest hop path: branch scenario

**Scenario 01/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['branch-edge-01', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [912]:
# Example 0911: Breadth-first shortest hop path - branch
case = {'hostname': 'branch-edge-01'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['branch-edge-01', 'core', 'exit']


### Example 0912 - Breadth-first shortest hop path: campus scenario

**Scenario 02/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['campus-access-02', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [913]:
# Example 0912: Breadth-first shortest hop path - campus
case = {'hostname': 'campus-access-02'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['campus-access-02', 'core', 'exit']


### Example 0913 - Breadth-first shortest hop path: dc scenario

**Scenario 03/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['dc-leaf-03', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [914]:
# Example 0913: Breadth-first shortest hop path - dc
case = {'hostname': 'dc-leaf-03'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['dc-leaf-03', 'core', 'exit']


### Example 0914 - Breadth-first shortest hop path: wan scenario

**Scenario 04/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wan-border-04', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [915]:
# Example 0914: Breadth-first shortest hop path - wan
case = {'hostname': 'wan-border-04'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['wan-border-04', 'core', 'exit']


### Example 0915 - Breadth-first shortest hop path: lab scenario

**Scenario 05/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['lab-router-05', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [916]:
# Example 0915: Breadth-first shortest hop path - lab
case = {'hostname': 'lab-router-05'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['lab-router-05', 'core', 'exit']


### Example 0916 - Breadth-first shortest hop path: wireless scenario

**Scenario 06/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wireless-dist-06', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [917]:
# Example 0916: Breadth-first shortest hop path - wireless
case = {'hostname': 'wireless-dist-06'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['wireless-dist-06', 'core', 'exit']

### Example 0917 - Breadth-first shortest hop path: remote scenario

**Scenario 07/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['remote-gateway-07', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [918]:
# Example 0917: Breadth-first shortest hop path - remote
case = {'hostname': 'remote-gateway-07'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['remote-gateway-07', 'core', 'exit']


### Example 0918 - Breadth-first shortest hop path: core scenario

**Scenario 08/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['core-router-08', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [919]:
# Example 0918: Breadth-first shortest hop path - core
case = {'hostname': 'core-router-08'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['core-router-08', 'core', 'exit']

### Example 0919 - Breadth-first shortest hop path: mgmt scenario

**Scenario 09/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mgmt-switch-09', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [920]:
# Example 0919: Breadth-first shortest hop path - mgmt
case = {'hostname': 'mgmt-switch-09'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['mgmt-switch-09', 'core', 'exit']


### Example 0920 - Breadth-first shortest hop path: backup scenario

**Scenario 10/10; pattern 092; Advanced.** This is a self-contained offline exercise. An offline topology model can explain a shortest unweighted path.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Put the source path in a deque. Return the first destination path or raise when unreachable. Follow the class-specific steps under Pattern 092 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['backup-edge-10', 'core', 'exit']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fewest hops is not lowest metric, highest bandwidth, or a routing protocol decision.

In [921]:
# Example 0920: Breadth-first shortest hop path - backup
case = {'hostname': 'backup-edge-10'}

from collections import deque

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def path(self, source, target):
        queue, visited = deque([[source]]), {source}
        while queue:
            path = queue.popleft()
            if path[-1] == target:
                return path
            for neighbor in sorted(self.adjacency.get(path[-1], ())):
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(path + [neighbor])
        raise LookupError("no modeled path")

host = case["hostname"]
topology = Topology({host: {"core"}, "core": {host, "exit"}, "exit": {"core"}})
assert topology.path(host, "exit") == [host, "core", "exit"]
print(topology.path(host, "exit"))


['backup-edge-10', 'core', 'exit']


## Pattern 093: Undirected cycle detection

**Core concept:** A DFS distinguishes a parent edge from a genuine previously visited neighbor.

**Network-engineering purpose:** A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

### Read the implementation in this order
1. Keep one visited set across graph components.
2. Track the parent in each recursive visit.
3. Skip only the immediate parent edge.
4. Compare a chain with a triangle.

**Common mistake / boundary:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

**Practice after the ten worked scenarios:** Add disconnected components and verify a cycle is found outside the first component.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0921 - Undirected cycle detection: branch scenario

**Scenario 01/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [922]:
# Example 0921: Undirected cycle detection - branch
case = {'hostname': 'branch-edge-01'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0922 - Undirected cycle detection: campus scenario

**Scenario 02/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [923]:
# Example 0922: Undirected cycle detection - campus
case = {'hostname': 'campus-access-02'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain:

 False triangle: True


### Example 0923 - Undirected cycle detection: dc scenario

**Scenario 03/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [924]:
# Example 0923: Undirected cycle detection - dc
case = {'hostname': 'dc-leaf-03'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0924 - Undirected cycle detection: wan scenario

**Scenario 04/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [925]:
# Example 0924: Undirected cycle detection - wan
case = {'hostname': 'wan-border-04'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0925 - Undirected cycle detection: lab scenario

**Scenario 05/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [926]:
# Example 0925: Undirected cycle detection - lab
case = {'hostname': 'lab-router-05'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0926 - Undirected cycle detection: wireless scenario

**Scenario 06/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [927]:
# Example 0926: Undirected cycle detection - wireless
case = {'hostname': 'wireless-dist-06'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0927 - Undirected cycle detection: remote scenario

**Scenario 07/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [928]:
# Example 0927: Undirected cycle detection - remote
case = {'hostname': 'remote-gateway-07'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain:

 False triangle: True


### Example 0928 - Undirected cycle detection: core scenario

**Scenario 08/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [929]:
# Example 0928: Undirected cycle detection - core
case = {'hostname': 'core-router-08'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0929 - Undirected cycle detection: mgmt scenario

**Scenario 09/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [930]:
# Example 0929: Undirected cycle detection - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


### Example 0930 - Undirected cycle detection: backup scenario

**Scenario 10/10; pattern 093; Advanced.** This is a self-contained offline exercise. A lab topology checker can flag a structural cycle without claiming a real forwarding loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Keep one visited set across graph components. Compare a chain with a triangle. Follow the class-specific steps under Pattern 093 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
chain: False triangle: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This algorithm assumes a simple undirected graph; use an appropriate algorithm for directed or multigraph models.

In [931]:
# Example 0930: Undirected cycle detection - backup
case = {'hostname': 'backup-edge-10'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = adjacency

    def has_cycle(self):
        visited = set()

        def visit(node, parent):
            visited.add(node)
            for neighbor in sorted(self.adjacency[node]):
                if neighbor == parent:
                    continue
                if neighbor in visited or visit(neighbor, node):
                    return True
            return False

        return any(visit(node, None) for node in self.adjacency if node not in visited)

host = case["hostname"]
chain = Topology({host: {"b"}, "b": {host, "c"}, "c": {"b"}})
triangle = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert chain.has_cycle() is False and triangle.has_cycle() is True
print("chain:", chain.has_cycle(), "triangle:", triangle.has_cycle())


chain: False triangle: True


## Pattern 094: Single-link failure reachability

**Core concept:** Analyze a copied model so a what-if scenario does not mutate the baseline.

**Network-engineering purpose:** A redundant lab triangle can retain connectivity after one modeled link is removed.

### Read the implementation in this order
1. Copy all adjacency sets.
2. Remove both directions of the selected link.
3. Search from source to target.
4. Assert reachability and prove baseline links remain present.

**Common mistake / boundary:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

**Practice after the ten worked scenarios:** Enumerate every link once and report which failures partition the topology.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0931 - Single-link failure reachability: branch scenario

**Scenario 01/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [932]:
# Example 0931: Single-link failure reachability - branch
case = {'hostname': 'branch-edge-01'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0932 - Single-link failure reachability: campus scenario

**Scenario 02/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [933]:
# Example 0932: Single-link failure reachability - campus
case = {'hostname': 'campus-access-02'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0933 - Single-link failure reachability: dc scenario

**Scenario 03/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [934]:
# Example 0933: Single-link failure reachability - dc
case = {'hostname': 'dc-leaf-03'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0934 - Single-link failure reachability: wan scenario

**Scenario 04/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [935]:
# Example 0934: Single-link failure reachability - wan
case = {'hostname': 'wan-border-04'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0935 - Single-link failure reachability: lab scenario

**Scenario 05/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [936]:
# Example 0935: Single-link failure reachability - lab
case = {'hostname': 'lab-router-05'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0936 - Single-link failure reachability: wireless scenario

**Scenario 06/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [937]:
# Example 0936: Single-link failure reachability - wireless
case = {'hostname': 'wireless-dist-06'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0937 - Single-link failure reachability: remote scenario

**Scenario 07/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [938]:
# Example 0937: Single-link failure reachability - remote
case = {'hostname': 'remote-gateway-07'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0938 - Single-link failure reachability: core scenario

**Scenario 08/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [939]:
# Example 0938: Single-link failure reachability - core
case = {'hostname': 'core-router-08'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0939 - Single-link failure reachability: mgmt scenario

**Scenario 09/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [940]:
# Example 0939: Single-link failure reachability - mgmt
case = {'hostname': 'mgmt-switch-09'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure


### Example 0940 - Single-link failure reachability: backup scenario

**Scenario 10/10; pattern 094; Advanced.** This is a self-contained offline exercise. A redundant lab triangle can retain connectivity after one modeled link is removed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Copy all adjacency sets. Assert reachability and prove baseline links remain present. Follow the class-specific steps under Pattern 094 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
modeled path survives one link failure
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This checks graph reachability only, not capacity, convergence time, or shared-risk failures.

In [941]:
# Example 0940: Single-link failure reachability - backup
case = {'hostname': 'backup-edge-10'}

class Topology:
    def __init__(self, adjacency):
        self.adjacency = {node: set(neighbors) for node, neighbors in adjacency.items()}

    def reachable_without(self, source, target, failed):
        graph = {node: set(neighbors) for node, neighbors in self.adjacency.items()}
        first, second = failed
        graph[first].discard(second)
        graph[second].discard(first)
        pending, visited = [source], set()
        while pending:
            node = pending.pop()
            if node == target:
                return True
            if node not in visited:
                visited.add(node)
                pending.extend(graph[node] - visited)
        return False

host = case["hostname"]
topology = Topology({host: {"b", "c"}, "b": {host, "c"}, "c": {host, "b"}})
assert topology.reachable_without(host, "b", (host, "b"))
assert "b" in topology.adjacency[host]
print("modeled path survives one link failure")


modeled path survives one link failure

## Pattern 095: Scoped address-plan overlap detection

**Core concept:** Network values can reveal unintended overlap within one declared allocation scope.

**Network-engineering purpose:** An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

### Read the implementation in this order
1. Parse and store one baseline prefix.
2. Parse each candidate.
3. Use overlaps for compatible address families.
4. Check one nested prefix and one unrelated documentation prefix.

**Common mistake / boundary:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

**Practice after the ten worked scenarios:** Include a VRF key and compare only allocations in the same scope.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0941 - Scoped address-plan overlap detection: branch scenario

**Scenario 01/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.0/28'` |
| `ip` | `'192.0.2.3'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [942]:
# Example 0941: Scoped address-plan overlap detection - branch
case = {'prefix': '192.0.2.0/28', 'ip': '192.0.2.3'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps:

True


### Example 0942 - Scoped address-plan overlap detection: campus scenario

**Scenario 02/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.16/28'` |
| `ip` | `'192.0.2.19'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [943]:
# Example 0942: Scoped address-plan overlap detection - campus
case = {'prefix': '192.0.2.16/28', 'ip': '192.0.2.19'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps:

 True


### Example 0943 - Scoped address-plan overlap detection: dc scenario

**Scenario 03/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.32/28'` |
| `ip` | `'192.0.2.35'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [944]:
# Example 0943: Scoped address-plan overlap detection - dc
case = {'prefix': '192.0.2.32/28', 'ip': '192.0.2.35'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0944 - Scoped address-plan overlap detection: wan scenario

**Scenario 04/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.48/28'` |
| `ip` | `'192.0.2.51'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [945]:
# Example 0944: Scoped address-plan overlap detection - wan
case = {'prefix': '192.0.2.48/28', 'ip': '192.0.2.51'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0945 - Scoped address-plan overlap detection: lab scenario

**Scenario 05/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.64/28'` |
| `ip` | `'192.0.2.67'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [946]:
# Example 0945: Scoped address-plan overlap detection - lab
case = {'prefix': '192.0.2.64/28', 'ip': '192.0.2.67'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0946 - Scoped address-plan overlap detection: wireless scenario

**Scenario 06/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.80/28'` |
| `ip` | `'192.0.2.83'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [947]:
# Example 0946: Scoped address-plan overlap detection - wireless
case = {'prefix': '192.0.2.80/28', 'ip': '192.0.2.83'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0947 - Scoped address-plan overlap detection: remote scenario

**Scenario 07/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.96/28'` |
| `ip` | `'192.0.2.99'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [948]:
# Example 0947: Scoped address-plan overlap detection - remote
case = {'prefix': '192.0.2.96/28', 'ip': '192.0.2.99'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0948 - Scoped address-plan overlap detection: core scenario

**Scenario 08/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.112/28'` |
| `ip` | `'192.0.2.115'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [949]:
# Example 0948: Scoped address-plan overlap detection - core
case = {'prefix': '192.0.2.112/28', 'ip': '192.0.2.115'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0949 - Scoped address-plan overlap detection: mgmt scenario

**Scenario 09/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.128/28'` |
| `ip` | `'192.0.2.131'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [950]:
# Example 0949: Scoped address-plan overlap detection - mgmt
case = {'prefix': '192.0.2.128/28', 'ip': '192.0.2.131'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Example 0950 - Scoped address-plan overlap detection: backup scenario

**Scenario 10/10; pattern 095; Advanced.** This is a self-contained offline exercise. An IPAM teaching object can distinguish nested allocations from unrelated prefixes.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `prefix` | `'192.0.2.144/28'` |
| `ip` | `'192.0.2.147'` |

**Logic to trace:** Parse and store one baseline prefix. Check one nested prefix and one unrelated documentation prefix. Follow the class-specific steps under Pattern 095 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
nested allocation overlaps: True
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Overlap is a policy question: nested routing prefixes can be intentional, and different VRFs may reuse addresses.

In [951]:
# Example 0950: Scoped address-plan overlap detection - backup
case = {'prefix': '192.0.2.144/28', 'ip': '192.0.2.147'}

from ipaddress import ip_network

class Allocation:
    def __init__(self, prefix):
        self.network = ip_network(prefix)

    def overlaps(self, prefix):
        other = ip_network(prefix)
        if self.network.version != other.version:
            return False
        return self.network.overlaps(other)

allocation = Allocation(case["prefix"])
assert allocation.overlaps(case["ip"] + "/32")
assert not allocation.overlaps("203.0.113.0/24")
print("nested allocation overlaps:", allocation.overlaps(case["ip"] + "/32"))


nested allocation overlaps: True


### Chapter 19 review

**Explain without running:** Explain the difference between modeled connectivity and actual forwarding reachability.

**Hands-on review:** pick one pattern from this chapter, change its scenario to a boundary condition, predict the output, and add one assertion that would catch an incorrect implementation. Name the invariant, the object that owns its state, and the exception behavior. Then explain which parts are offline teaching assumptions and which parts would need a reviewed real-device adapter.

<a id="chapter-20"></a>
# Chapter 20: Integrated network-automation capstones

**Level:** Advanced

**Learning goal:** Combine the models into reviewable offline workflows with explicit success and failure outcomes.

**Capstone architecture.** Keep domain models, transport boundaries, policies, orchestration,
and reporting distinct. The following worked patterns combine multiple earlier concepts
rather than introducing a giant framework. All data and outcomes are synthetic. No command
is sent to a real device, no package installation is needed, and no credentials are used.

**A reviewable change flow.** Validate intent -> observe a baseline -> compute a plan -> obtain
approval for that exact plan -> apply through a controlled adapter -> collect fresh state ->
verify postconditions -> record an accurate outcome. If application or verification fails,
attempt the documented recovery and report whether recovery itself succeeded.

**Fleet outcomes are per-device.** One failed read must not become an empty successful report.
Keep target identity with the result, distinguish expected transport errors from programming
errors, and bound concurrency. Preserve enough diagnostics for the operator without copying
sensitive payloads into logs.

**Telemetry pipelines have units and freshness.** A calculation is only meaningful if samples
belong to the same interface identity, counters are comparable, and timestamps are trustworthy.
A compliance engine should distinguish violated policy from missing or malformed evidence.

**Beyond this notebook.** A real project needs supported library versions, reviewed platform
adapters, authorization, secrets management, reliable persistence, CI, observability, and
integration tests in an approved lab. These notes teach object design and reasoning. They do
not replace vendor documentation or an organization's change process.

For each capstone, sketch responsibilities first, run all ten datasets, identify the invariants,
then implement the extension prompt. Prefer a small explicit design you can explain to another
engineer over clever metaprogramming.

**Pause and explain:** Walk one capstone from input through result and identify the exact trust and side-effect boundaries.

## Pattern 096: Capstone: reviewed idempotent MTU deployment

**Core concept:** Combine an immutable intent, a plan, approval, reconciliation, and postcheck.

**Network-engineering purpose:** An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

### Read the implementation in this order
1. Represent desired MTU with a frozen dataclass.
2. Compute whether baseline differs.
3. Require approval before any mutation.
4. Verify the postcondition and check a repeated approved apply is unchanged.

**Common mistake / boundary:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

**Practice after the ten worked scenarios:** Bind approval to baseline plus intent and reject drift between plan and apply.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0951 - Capstone: reviewed idempotent MTU deployment: branch scenario

**Scenario 01/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['unchanged', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [952]:
# Example 0951: Capstone: reviewed idempotent MTU deployment - branch
case = {'mtu': 1500, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['unchanged', 'unchanged']


### Example 0952 - Capstone: reviewed idempotent MTU deployment: campus scenario

**Scenario 02/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |
| `approved` | `False` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['denied']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [953]:
# Example 0952: Capstone: reviewed idempotent MTU deployment - campus
case = {'mtu': 9000, 'approved': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['denied']


### Example 0953 - Capstone: reviewed idempotent MTU deployment: dc scenario

**Scenario 03/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 9216 audit: ['changed', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [954]:
# Example 0953: Capstone: reviewed idempotent MTU deployment - dc
case = {'mtu': 9216, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 9216 audit: ['changed', 'unchanged']


### Example 0954 - Capstone: reviewed idempotent MTU deployment: wan scenario

**Scenario 04/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 576 audit: ['changed', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [955]:
# Example 0954: Capstone: reviewed idempotent MTU deployment - wan
case = {'mtu': 576, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 576 audit: ['changed', 'unchanged']


### Example 0955 - Capstone: reviewed idempotent MTU deployment: lab scenario

**Scenario 05/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |
| `approved` | `False` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['denied']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [956]:
# Example 0955: Capstone: reviewed idempotent MTU deployment - lab
case = {'mtu': 1600, 'approved': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['denied']


### Example 0956 - Capstone: reviewed idempotent MTU deployment: wireless scenario

**Scenario 06/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['unchanged', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [957]:
# Example 0956: Capstone: reviewed idempotent MTU deployment - wireless
case = {'mtu': 1500, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed:

 1500 audit: ['unchanged', 'unchanged']


### Example 0957 - Capstone: reviewed idempotent MTU deployment: remote scenario

**Scenario 07/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |
| `approved` | `False` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['denied']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [958]:
# Example 0957: Capstone: reviewed idempotent MTU deployment - remote
case = {'mtu': 2000, 'approved': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['denied']


### Example 0958 - Capstone: reviewed idempotent MTU deployment: core scenario

**Scenario 08/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 4470 audit: ['changed', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [959]:
# Example 0958: Capstone: reviewed idempotent MTU deployment - core
case = {'mtu': 4470, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 4470 audit: ['changed', 'unchanged']


### Example 0959 - Capstone: reviewed idempotent MTU deployment: mgmt scenario

**Scenario 09/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |
| `approved` | `False` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['denied']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [960]:
# Example 0959: Capstone: reviewed idempotent MTU deployment - mgmt
case = {'mtu': 9100, 'approved': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['denied']


### Example 0960 - Capstone: reviewed idempotent MTU deployment: backup scenario

**Scenario 10/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 8192 audit: ['changed', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [961]:
# Example 0960: Capstone: reviewed idempotent MTU deployment - backup
case = {'mtu': 8192, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 8192 audit: ['changed', 'unchanged']


## Pattern 097: Capstone: fleet collection with structured outcomes

**Core concept:** A result object preserves target identity and explicit transport failures.

**Network-engineering purpose:** Fleet reporting should not hide one failed read behind an aggregate success count.

### Read the implementation in this order
1. Use a frozen outcome record.
2. Inject a replay transport.
3. Catch only the expected TimeoutError per target.
4. Assert the successful and failed result shapes separately.

**Common mistake / boundary:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

**Practice after the ten worked scenarios:** Add bounded concurrency while preserving a stable input-to-output mapping.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0961 - Capstone: fleet collection with structured outcomes: branch scenario

**Scenario 01/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='branch-edge-01', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [962]:
# Example 0961: Capstone: fleet collection with structured outcomes - branch
case = {'hostname': 'branch-edge-01'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='branch-edge-01', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0962 - Capstone: fleet collection with structured outcomes: campus scenario

**Scenario 02/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='campus-access-02', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [963]:
# Example 0962: Capstone: fleet collection with structured outcomes - campus
case = {'hostname': 'campus-access-02'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='campus-access-02', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0963 - Capstone: fleet collection with structured outcomes: dc scenario

**Scenario 03/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='dc-leaf-03', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [964]:
# Example 0963: Capstone: fleet collection with structured outcomes - dc
case = {'hostname': 'dc-leaf-03'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='dc-leaf-03', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0964 - Capstone: fleet collection with structured outcomes: wan scenario

**Scenario 04/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='wan-border-04', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [965]:
# Example 0964: Capstone: fleet collection with structured outcomes - wan
case = {'hostname': 'wan-border-04'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='wan-border-04', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]

### Example 0965 - Capstone: fleet collection with structured outcomes: lab scenario

**Scenario 05/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='lab-router-05', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [966]:
# Example 0965: Capstone: fleet collection with structured outcomes - lab
case = {'hostname': 'lab-router-05'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='lab-router-05', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]

### Example 0966 - Capstone: fleet collection with structured outcomes: wireless scenario

**Scenario 06/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='wireless-dist-06', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [967]:
# Example 0966: Capstone: fleet collection with structured outcomes - wireless
case = {'hostname': 'wireless-dist-06'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='wireless-dist-06', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0967 - Capstone: fleet collection with structured outcomes: remote scenario

**Scenario 07/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='remote-gateway-07', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [968]:
# Example 0967: Capstone: fleet collection with structured outcomes - remote
case = {'hostname': 'remote-gateway-07'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='remote-gateway-07', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0968 - Capstone: fleet collection with structured outcomes: core scenario

**Scenario 08/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='core-router-08', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [969]:
# Example 0968: Capstone: fleet collection with structured outcomes - core
case = {'hostname': 'core-router-08'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='core-router-08', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]

### Example 0969 - Capstone: fleet collection with structured outcomes: mgmt scenario

**Scenario 09/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='mgmt-switch-09', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [970]:
# Example 0969: Capstone: fleet collection with structured outcomes - mgmt
case = {'hostname': 'mgmt-switch-09'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='mgmt-switch-09', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0970 - Capstone: fleet collection with structured outcomes: backup scenario

**Scenario 10/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='backup-edge-10', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [971]:
# Example 0970: Capstone: fleet collection with structured outcomes - backup
case = {'hostname': 'backup-edge-10'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='backup-edge-10', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]

## Pattern 098: Capstone: rate calculation and health policy

**Core concept:** Separate immutable samples, unit-aware calculation, and threshold evaluation.

**Network-engineering purpose:** A telemetry report can show both the measured bps value and the policy result.

### Read the implementation in this order
1. Represent each timestamped counter sample.
2. Reject non-increasing time and counter resets.
3. Calculate bits per second.
4. Evaluate against an injected policy limit and assert both data and classification.

**Common mistake / boundary:** These synthetic samples assume no wrap, no reset, and the same interface identity.

**Practice after the ten worked scenarios:** Add a result state for unavailable evidence without converting it to a healthy zero rate.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0971 - Capstone: rate calculation and health policy: branch scenario

**Scenario 01/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `1000` |
| `traffic_bytes` | `0` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 0.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [972]:
# Example 0971: Capstone: rate calculation and health policy - branch
case = {'counter': 1000, 'traffic_bytes': 0}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 0.0 health: ok


### Example 0972 - Capstone: rate calculation and health policy: campus scenario

**Scenario 02/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `2000` |
| `traffic_bytes` | `500` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 2000.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [973]:
# Example 0972: Capstone: rate calculation and health policy - campus
case = {'counter': 2000, 'traffic_bytes': 500}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 2000.0 health: ok


### Example 0973 - Capstone: rate calculation and health policy: dc scenario

**Scenario 03/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `3000` |
| `traffic_bytes` | `1000` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 4000.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [974]:
# Example 0973: Capstone: rate calculation and health policy - dc
case = {'counter': 3000, 'traffic_bytes': 1000}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 4000.0 health: ok


### Example 0974 - Capstone: rate calculation and health policy: wan scenario

**Scenario 04/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `4000` |
| `traffic_bytes` | `1001` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 4004.0 health: high
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [975]:
# Example 0974: Capstone: rate calculation and health policy - wan
case = {'counter': 4000, 'traffic_bytes': 1001}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 4004.0 health: high


### Example 0975 - Capstone: rate calculation and health policy: lab scenario

**Scenario 05/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `5000` |
| `traffic_bytes` | `2000` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 8000.0 health: high
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [976]:
# Example 0975: Capstone: rate calculation and health policy - lab
case = {'counter': 5000, 'traffic_bytes': 2000}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 8000.0 health: high


### Example 0976 - Capstone: rate calculation and health policy: wireless scenario

**Scenario 06/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `6000` |
| `traffic_bytes` | `250` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 1000.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [977]:
# Example 0976: Capstone: rate calculation and health policy - wireless
case = {'counter': 6000, 'traffic_bytes': 250}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 1000.0 health: ok


### Example 0977 - Capstone: rate calculation and health policy: remote scenario

**Scenario 07/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `7000` |
| `traffic_bytes` | `3000` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 12000.0 health: high
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [978]:
# Example 0977: Capstone: rate calculation and health policy - remote
case = {'counter': 7000, 'traffic_bytes': 3000}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 12000.0 health: high


### Example 0978 - Capstone: rate calculation and health policy: core scenario

**Scenario 08/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `8000` |
| `traffic_bytes` | `750` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 3000.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [979]:
# Example 0978: Capstone: rate calculation and health policy - core
case = {'counter': 8000, 'traffic_bytes': 750}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 3000.0 health: ok


### Example 0979 - Capstone: rate calculation and health policy: mgmt scenario

**Scenario 09/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `9000` |
| `traffic_bytes` | `1500` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 6000.0 health: high
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [980]:
# Example 0979: Capstone: rate calculation and health policy - mgmt
case = {'counter': 9000, 'traffic_bytes': 1500}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 6000.0 health: high


### Example 0980 - Capstone: rate calculation and health policy: backup scenario

**Scenario 10/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `10000` |
| `traffic_bytes` | `100` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 400.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [981]:
# Example 0980: Capstone: rate calculation and health policy - backup
case = {'counter': 10000, 'traffic_bytes': 100}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 400.0 health: ok


## Pattern 099: Capstone: composable compliance evidence

**Core concept:** Individual checks return explicit evidence and an engine aggregates it.

**Network-engineering purpose:** An interface can fail one requirement without losing the evidence for checks that passed.

### Read the implementation in this order
1. Define a frozen check result.
2. Implement separate MTU and admin-state checks.
3. Inject checks into an engine.
4. Verify each result and the overall all-pass condition.

**Common mistake / boundary:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

**Practice after the ten worked scenarios:** Extend the result with PASS/FAIL/UNKNOWN and define aggregation semantics.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0981 - Capstone: composable compliance evidence: branch scenario

**Scenario 01/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=1500), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [982]:
# Example 0981: Capstone: composable compliance evidence - branch
case = {'mtu': 1500, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=1500), Evidence(check='admin', passed=True, observed=True)]


### Example 0982 - Capstone: composable compliance evidence: campus scenario

**Scenario 02/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |
| `healthy` | `False` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=9000), Evidence(check='admin', passed=False, observed=False)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [983]:
# Example 0982: Capstone: composable compliance evidence - campus
case = {'mtu': 9000, 'healthy': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=9000), Evidence(check='admin', passed=False, observed=False)]


### Example 0983 - Capstone: composable compliance evidence: dc scenario

**Scenario 03/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=9216), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [984]:
# Example 0983: Capstone: composable compliance evidence - dc
case = {'mtu': 9216, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=9216), Evidence(check='admin', passed=True, observed=True)]

### Example 0984 - Capstone: composable compliance evidence: wan scenario

**Scenario 04/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=False, observed=576), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [985]:
# Example 0984: Capstone: composable compliance evidence - wan
case = {'mtu': 576, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=False, observed=576), Evidence(check='admin', passed=True, observed=True)]

### Example 0985 - Capstone: composable compliance evidence: lab scenario

**Scenario 05/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |
| `healthy` | `False` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=1600), Evidence(check='admin', passed=False, observed=False)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [986]:
# Example 0985: Capstone: composable compliance evidence - lab
case = {'mtu': 1600, 'healthy': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=1600), Evidence(check='admin', passed=False, observed=False)]

### Example 0986 - Capstone: composable compliance evidence: wireless scenario

**Scenario 06/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=1500), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [987]:
# Example 0986: Capstone: composable compliance evidence - wireless
case = {'mtu': 1500, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=1500), Evidence(check='admin', passed=True, observed=True)]


### Example 0987 - Capstone: composable compliance evidence: remote scenario

**Scenario 07/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |
| `healthy` | `False` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=2000), Evidence(check='admin', passed=False, observed=False)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [988]:
# Example 0987: Capstone: composable compliance evidence - remote
case = {'mtu': 2000, 'healthy': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=2000), Evidence(check='admin', passed=False, observed=False)]


### Example 0988 - Capstone: composable compliance evidence: core scenario

**Scenario 08/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=4470), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [989]:
# Example 0988: Capstone: composable compliance evidence - core
case = {'mtu': 4470, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=4470), Evidence(check='admin', passed=True, observed=True)]


### Example 0989 - Capstone: composable compliance evidence: mgmt scenario

**Scenario 09/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |
| `healthy` | `False` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=9100), Evidence(check='admin', passed=False, observed=False)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [990]:
# Example 0989: Capstone: composable compliance evidence - mgmt
case = {'mtu': 9100, 'healthy': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=9100), Evidence(check='admin', passed=False, observed=False)]


### Example 0990 - Capstone: composable compliance evidence: backup scenario

**Scenario 10/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=8192), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [991]:
# Example 0990: Capstone: composable compliance evidence - backup
case = {'mtu': 8192, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=8192), Evidence(check='admin', passed=True, observed=True)]


## Pattern 100: Capstone: validated inventory import and report

**Core concept:** A schema boundary, immutable record, duplicate check, and report form one small pipeline.

**Network-engineering purpose:** A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

### Read the implementation in this order
1. Validate field names and nonempty strings.
2. Create a frozen record through from_mapping.
3. Reject a duplicate before repository mutation.
4. Export a deterministic public report.

**Common mistake / boundary:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

**Practice after the ten worked scenarios:** Add address validation and a versioned JSON import/export format with round-trip tests.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0991 - Capstone: validated inventory import and report: branch scenario

**Scenario 01/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'branch'` |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "branch-edge-01", "site": "branch"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [992]:
# Example 0991: Capstone: validated inventory import and report - branch
case = {'site': 'branch', 'hostname': 'branch-edge-01'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "branch-edge-01", "site": "branch"}]


### Example 0992 - Capstone: validated inventory import and report: campus scenario

**Scenario 02/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'campus'` |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "campus-access-02", "site": "campus"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [993]:
# Example 0992: Capstone: validated inventory import and report - campus
case = {'site': 'campus', 'hostname': 'campus-access-02'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "campus-access-02", "site": "campus"}]


### Example 0993 - Capstone: validated inventory import and report: dc scenario

**Scenario 03/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'dc'` |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "dc-leaf-03", "site": "dc"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [994]:
# Example 0993: Capstone: validated inventory import and report - dc
case = {'site': 'dc', 'hostname': 'dc-leaf-03'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "dc-leaf-03", "site": "dc"}]


### Example 0994 - Capstone: validated inventory import and report: wan scenario

**Scenario 04/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wan'` |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "wan-border-04", "site": "wan"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [995]:
# Example 0994: Capstone: validated inventory import and report - wan
case = {'site': 'wan', 'hostname': 'wan-border-04'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "wan-border-04", "site": "wan"}]


### Example 0995 - Capstone: validated inventory import and report: lab scenario

**Scenario 05/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'lab'` |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "lab-router-05", "site": "lab"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [996]:
# Example 0995: Capstone: validated inventory import and report - lab
case = {'site': 'lab', 'hostname': 'lab-router-05'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "lab-router-05", "site": "lab"}]


### Example 0996 - Capstone: validated inventory import and report: wireless scenario

**Scenario 06/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wireless'` |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "wireless-dist-06", "site": "wireless"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [997]:
# Example 0996: Capstone: validated inventory import and report - wireless
case = {'site': 'wireless', 'hostname': 'wireless-dist-06'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "wireless-dist-06", "site": "wireless"}]


### Example 0997 - Capstone: validated inventory import and report: remote scenario

**Scenario 07/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'remote'` |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "remote-gateway-07", "site": "remote"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [998]:
# Example 0997: Capstone: validated inventory import and report - remote
case = {'site': 'remote', 'hostname': 'remote-gateway-07'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "remote-gateway-07", "site": "remote"}]


### Example 0998 - Capstone: validated inventory import and report: core scenario

**Scenario 08/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'core'` |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "core-router-08", "site": "core"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [999]:
# Example 0998: Capstone: validated inventory import and report - core
case = {'site': 'core', 'hostname': 'core-router-08'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "core-router-08", "site": "core"}]


### Example 0999 - Capstone: validated inventory import and report: mgmt scenario

**Scenario 09/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'mgmt'` |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "mgmt-switch-09", "site": "mgmt"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [1000]:
# Example 0999: Capstone: validated inventory import and report - mgmt
case = {'site': 'mgmt', 'hostname': 'mgmt-switch-09'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "mgmt-switch-09", "site": "mgmt"}]


### Example 1000 - Capstone: validated inventory import and report: backup scenario

**Scenario 10/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'backup'` |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "backup-edge-10", "site": "backup"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [1001]:
# Example 1000: Capstone: validated inventory import and report - backup
case = {'site': 'backup', 'hostname': 'backup-edge-10'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "backup-edge-10", "site": "backup"}]


### Chapter 20 review

**Explain without running:** Walk one capstone from input through result and identify the exact trust and side-effect boundaries.

**Hands-on review:** pick one pattern from this chapter, change its scenario to a boundary condition, predict the output, and add one assertion that would catch an incorrect implementation. Name the invariant, the object that owns its state, and the exception behavior. Then explain which parts are offline teaching assumptions and which parts would need a reviewed real-device adapter.


## References, production boundaries, and study checkpoints

These are original worked teaching examples. The authoritative references below are for checking
Python's language and library contracts, not vendor command support. The documentation series is
pinned to Python 3.13 rather than assuming the unversioned documentation matches your installed kernel.

| Subject | Official Python reference |
| --- | --- |
| Classes, attributes, inheritance, scope | `https://docs.python.org/3.13/tutorial/classes.html` |
| Special methods, descriptors, __new__, slots, metaclasses | `https://docs.python.org/3.13/reference/datamodel.html` |
| Dataclasses, frozen fields, factories, replace | `https://docs.python.org/3.13/library/dataclasses.html` |
| Protocol, Generic, type variables, runtime checks | `https://docs.python.org/3.13/library/typing.html` |
| ABC and abstract methods | `https://docs.python.org/3.13/library/abc.html` |
| Context-management helpers | `https://docs.python.org/3.13/library/contextlib.html` |
| Tasks, TaskGroup, cancellation | `https://docs.python.org/3.13/library/asyncio-task.html` |
| IP addresses, networks, interfaces | `https://docs.python.org/3.13/library/ipaddress.html` |
| Mock and autospec | `https://docs.python.org/3.13/library/unittest.mock.html` |
| Shallow and deep copying | `https://docs.python.org/3.13/library/copy.html` |
| cached_property and function wrappers | `https://docs.python.org/3.13/library/functools.html` |
| Named enumeration values | `https://docs.python.org/3.13/library/enum.html` |
| Thread pools and Futures | `https://docs.python.org/3.13/library/concurrent.futures.html` |
| Thread locks | `https://docs.python.org/3.13/library/threading.html` |
| deque and other containers | `https://docs.python.org/3.13/library/collections.html` |
| Heap operations | `https://docs.python.org/3.13/library/heapq.html` |
| Test cases and assertions | `https://docs.python.org/3.13/library/unittest.html` |
| JSON representation | `https://docs.python.org/3.13/library/json.html` |

### Checkpoints
**Beginner:** Explain every line of a small Device class, build two independent instances, write an
instance method, and correct a mutable-default bug without looking at the answer.

**Intermediate:** Design a validated interface value, inject a fake transport, compare ABC and
Protocol, justify composition versus inheritance, serialize explicit fields, and write a negative test
that proves state remains unchanged after failure.

**Advanced:** Design an idempotent approved plan, report per-device failures honestly, bound async
work, reason about resource lifetime and cancellation, state counter-reset assumptions, and keep
domain models independent of vendor libraries.

### Five independent projects to try after the worked examples
1. Build a versioned offline inventory importer with duplicate detection, IPv4/IPv6 validation, and
   explicit error reports per source record.
2. Build a simulated interface-intent planner with immutable plans, baseline drift detection,
   approval binding, rollback outcomes, and idempotent reruns.
3. Build a bounded fleet collector with a fake clock, timeout injection, per-target results, and
   deterministic tests; no real devices are needed.
4. Build a telemetry evaluator that distinguishes valid rate, reset, wrap ambiguity, stale data,
   and missing data without inventing zero-valued success results.
5. Build a scoped topology/IPAM report with graph assumptions, link-failure analysis, VRF-aware
   overlap rules, and reproducible public JSON output.

These projects are extension ideas, not part of the counted 1,000 worked examples.
